In [1]:


import sys
import os
import platform
import random
import json

import numpy as np
import pandas as pd

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("=" * 60)
print("HYBRID GNN-LLM REPRODUCIBILITY")
print("=" * 60)

print("\n[Python]")
print(sys.version)

print("\n[Platform]")
print(platform.platform())

print("\n[Core libraries]")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("\n[PyTorch]")
try:
    import torch

    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    print("Version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print(
            "GPU memory (GB):",
            round(
                torch.cuda.get_device_properties(0).total_memory
                / (1024 ** 3),
                2
            )
        )
    else:
        print("GPU: Not available")

except ImportError:
    print("PyTorch is not installed")

print("\n[Working directory]")
print(os.getcwd())

print("\n[Audit complete]")


HYBRID GNN-LLM REPRODUCIBILITY

[Python]
3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]

[Platform]
Linux-6.6.122+-x86_64-with-glibc2.39

[Core libraries]
NumPy: 2.1.3
Pandas: 2.2.3

[PyTorch]
Version: 2.11.0+cpu
CUDA available: False
GPU: Not available

[Working directory]
/content

[Audit complete]


In [2]:
from pathlib import Path
import os

PROJECT_DIR = Path("/content/hybrid_gnn_llm")

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
CACHE_DIR = PROJECT_DIR / "cache"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"

for folder in [
    RAW_DIR,
    PROCESSED_DIR,
    CACHE_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders created.")
print("-" * 55)

for folder in PROJECT_DIR.rglob("*"):
    if folder.is_dir():
        print("DIR :", folder.relative_to(PROJECT_DIR))

print("\nFiles currently available in /content:")
print("-" * 55)

for item in sorted(Path("/content").iterdir()):
    if item.is_file():
        print(f"FILE: {item.name} ({item.stat().st_size:,} bytes)")
    elif item.is_dir():
        print(f"DIR : {item.name}")

print("\nData folders ready:")
print("Raw data      :", RAW_DIR)
print("Processed data:", PROCESSED_DIR)
print("Cache         :", CACHE_DIR)
print("Results       :", RESULTS_DIR)


Project folders created.
-------------------------------------------------------
DIR : figures
DIR : results
DIR : data
DIR : cache
DIR : data/processed
DIR : data/raw

Files currently available in /content:
-------------------------------------------------------
DIR : .config
DIR : hybrid_gnn_llm
DIR : sample_data

Data folders ready:
Raw data      : /content/hybrid_gnn_llm/data/raw
Processed data: /content/hybrid_gnn_llm/data/processed
Cache         : /content/hybrid_gnn_llm/cache
Results       : /content/hybrid_gnn_llm/results


In [3]:

from pathlib import Path

SEARCH_ROOTS = [
    Path("/content/sample_data"),
    Path("/content/hybrid_gnn_llm/data/raw"),
]

SUPPORTED_EXTENSIONS = {
    ".csv", ".tsv", ".json", ".jsonl",
    ".parquet", ".xlsx", ".zip", ".gz",
    ".pkl", ".pickle"
}

print("Searching for candidate data files...")
print("=" * 65)

found_files = []

for root in SEARCH_ROOTS:
    print(f"\nDirectory: {root}")

    if not root.exists():
        print("  Directory does not exist.")
        continue

    matches = sorted(
        path for path in root.rglob("*")
        if path.is_file()
        and any(
            path.name.lower().endswith(ext)
            for ext in SUPPORTED_EXTENSIONS
        )
    )

    if not matches:
        print("  No supported data files found.")
    else:
        for path in matches:
            size_mb = path.stat().st_size / (1024 ** 2)
            print(f"  {path}  ({size_mb:.2f} MB)")
            found_files.append(str(path))

print("\n" + "=" * 65)
print("Total candidate files:", len(found_files))

if found_files:
    print("\nCandidate file paths:")
    for path in found_files:
        print(path)
else:
    print(
        "\nNo dataset files found. We need to obtain the original "
        "public dataset files before model training."
    )


Searching for candidate data files...

Directory: /content/sample_data
  /content/sample_data/anscombe.json  (0.00 MB)
  /content/sample_data/california_housing_test.csv  (0.29 MB)
  /content/sample_data/california_housing_train.csv  (1.63 MB)
  /content/sample_data/mnist_test.csv  (17.44 MB)
  /content/sample_data/mnist_train_small.csv  (34.83 MB)

Directory: /content/hybrid_gnn_llm/data/raw
  No supported data files found.

Total candidate files: 5

Candidate file paths:
/content/sample_data/anscombe.json
/content/sample_data/california_housing_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv
/content/sample_data/mnist_train_small.csv


In [4]:


from pathlib import Path
import shutil
import json

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
RAW_DIR = PROJECT_DIR / "data" / "raw"

disk = shutil.disk_usage("/content")

print("COLAB STORAGE")
print("=" * 60)
print(f"Total space : {disk.total / (1024**3):.2f} GB")
print(f"Used space  : {disk.used / (1024**3):.2f} GB")
print(f"Free space  : {disk.free / (1024**3):.2f} GB")

datasets = {
    "YelpCHI": {
        "status": "not_downloaded",
        "role": "primary fake-review classification candidate",
        "source": "https://doi.org/10.57760/sciencedb.17088",
        "label_verification": "pending",
        "local_directory": str(RAW_DIR / "yelpchi"),
    },
    "FakeNewsNet": {
        "status": "not_downloaded",
        "role": "fake-news classification candidate",
        "source": "https://github.com/KaiDMML/FakeNewsNet",
        "label_verification": "pending",
        "local_directory": str(RAW_DIR / "fakenewsnet"),
    },
    "Amazon Reviews 2023 - All Beauty": {
        "status": "not_downloaded",
        "role": "not approved for fake/genuine classification yet",
        "source": "https://amazon-reviews-2023.github.io/",
        "label_verification": (
            "must establish an independent fake/genuine label source"
        ),
        "local_directory": str(RAW_DIR / "amazon_reviews_2023"),
    },
}

for item in datasets.values():
    Path(item["local_directory"]).mkdir(
        parents=True, exist_ok=True
    )

registry_path = PROJECT_DIR / "dataset_registry.json"
registry_path.write_text(
    json.dumps(datasets, indent=2),
    encoding="utf-8"
)

print("\nDATASET REGISTRY")
print("=" * 60)

for name, info in datasets.items():
    print(f"\n{name}")
    print("  Status:", info["status"])
    print("  Role:", info["role"])
    print("  Label check:", info["label_verification"])
    print("  Folder:", info["local_directory"])

print("\nRegistry saved to:", registry_path)


COLAB STORAGE
Total space : 107.72 GB
Used space  : 20.49 GB
Free space  : 87.21 GB

DATASET REGISTRY

YelpCHI
  Status: not_downloaded
  Role: primary fake-review classification candidate
  Label check: pending
  Folder: /content/hybrid_gnn_llm/data/raw/yelpchi

FakeNewsNet
  Status: not_downloaded
  Role: fake-news classification candidate
  Label check: pending
  Folder: /content/hybrid_gnn_llm/data/raw/fakenewsnet

Amazon Reviews 2023 - All Beauty
  Status: not_downloaded
  Role: not approved for fake/genuine classification yet
  Label check: must establish an independent fake/genuine label source
  Folder: /content/hybrid_gnn_llm/data/raw/amazon_reviews_2023

Registry saved to: /content/hybrid_gnn_llm/dataset_registry.json


In [5]:

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

doi_url = "https://doi.org/10.57760/sciencedb.17088"

print("Checking cited YelpCHI source...")
print("=" * 70)

try:
    response = requests.get(
        doi_url,
        timeout=30,
        allow_redirects=True,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    print("HTTP status:", response.status_code)
    print("Resolved URL:", response.url)
    print("Content type:", response.headers.get("Content-Type"))

    response.raise_for_status()

    content_type = response.headers.get("Content-Type", "").lower()

    if "html" not in content_type:
        print("\nThe response is not an HTML landing page.")
        print("First 500 characters:")
        print(response.text[:500])
    else:
        soup = BeautifulSoup(response.text, "html.parser")

        print("\nPage title:")
        print(soup.title.get_text(" ", strip=True) if soup.title else "(none)")

        print("\nRelevant page text:")
        text = soup.get_text(" ", strip=True)

        for keyword in [
            "YelpCHI", "YelpChi", "download", "file",
            "dataset", "license", "citation"
        ]:
            pos = text.lower().find(keyword.lower())
            if pos >= 0:
                start = max(0, pos - 180)
                end = min(len(text), pos + 450)
                print(f"\n[{keyword}] {text[start:end]}")

        print("\nPotential dataset links:")
        seen = set()

        for a in soup.find_all("a", href=True):
            label = a.get_text(" ", strip=True)
            href = urljoin(response.url, a["href"])

            combined = (label + " " + href).lower()

            if any(
                word in combined
                for word in [
                    "download", "yelpchi", "yelp-chi",
                    ".zip", ".csv", ".rar", ".7z"
                ]
            ):
                if href not in seen:
                    seen.add(href)
                    print(f"- {label or '(no link text)'}: {href}")

except Exception as exc:
    print("Could not inspect the repository automatically.")
    print("Error:", repr(exc))
    print(
        "\nPlease open the DOI manually in your browser and share "
        "the dataset title and the listed downloadable filenames."
    )


Checking cited YelpCHI source...
HTTP status: 200
Resolved URL: https://www.scidb.cn/detail?dataSetId=76673646fb7241f58aada7b9f24b25fe
Content type: text/html; charset=utf-8

Page title:
Yelp Dataset

Relevant page text:

[dataset] Yelp Dataset

Potential dataset links:


In [6]:

import requests
import re
import json
from bs4 import BeautifulSoup

page_url = (
    "https://www.scidb.cn/detail"
    "?dataSetId=76673646fb7241f58aada7b9f24b25fe"
)

response = requests.get(
    page_url,
    timeout=30,
    headers={"User-Agent": "Mozilla/5.0"}
)
response.raise_for_status()

html = response.text
soup = BeautifulSoup(html, "html.parser")

print("PAGE METADATA")
print("=" * 65)

for tag in soup.find_all("meta"):
    name = tag.get("name") or tag.get("property")
    content = tag.get("content")

    if name and content:
        if any(
            word in name.lower()
            for word in ["title", "description", "citation", "dataset", "doi"]
        ):
            print(f"{name}: {content[:500]}")

print("\nEMBEDDED DATASET REFERENCES")
print("=" * 65)

patterns = [
    r"76673646fb7241f58aada7b9f24b25fe",
    r"download",
    r"fileList",
    r"fileUrl",
    r"dataSetId",
    r"YelpCHI",
    r"Yelp Dataset",
]

for pattern in patterns:
    matches = list(re.finditer(pattern, html, flags=re.IGNORECASE))
    print(f"\n{pattern}: {len(matches)} match(es)")

    for match in matches[:3]:
        start = max(0, match.start() - 180)
        end = min(len(html), match.end() + 350)
        snippet = re.sub(r"\s+", " ", html[start:end])
        print(snippet[:550])

print("\nPAGE SCRIPT SOURCES")
print("=" * 65)

for script in soup.find_all("script", src=True):
    print(script["src"])

print("\nInspection complete. No files have been downloaded.")


PAGE METADATA
description: This dataset is a publicly available fake review dataset from the Yelp website, including the YelpChi, YelpNYC, and YelpZIP datasets.

EMBEDDED DATASET REFERENCES

76673646fb7241f58aada7b9f24b25fe: 3 match(es)
"/css/base/base.css"><link data-n-head="ssr" rel="stylesheet" href="/css/base/animation.css"><link data-n-head="ssr" rel="canonical" href="https://www.scidb.cn/en/detail?dataSetId=76673646fb7241f58aada7b9f24b25fe"><style data-n-head="vuetify" type="text/css" id="vuetify-theme-stylesheet" nonce="undefined">.v-application a { color: #1976d2; } .v-application .primary { background-color: #1976d2 !important; border-color: #1976d2 !important; } .v-application .primary--text { color: #1976d2 !important; caret-color: #1976d2 !important; } .v-applicati
","subField":"cr:subField","transform":"cr:transform","wd":"https://www.wikidata.org/wiki/"},"@id":"https://doi.org/10.57760/sciencedb.17088","@type":"sc:Dataset","citeAs":"@data{ 76673646fb7241f58aada7b9f24b25fe

In [7]:


import requests
import re
from urllib.parse import urljoin

BASE_URL = "https://www.scidb.cn"
PAGE_URL = (
    BASE_URL
    + "/detail?dataSetId=76673646fb7241f58aada7b9f24b25fe"
)

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

response = session.get(PAGE_URL, timeout=30)
response.raise_for_status()

from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

scripts = [
    urljoin(BASE_URL, tag["src"])
    for tag in soup.find_all("script", src=True)
    if "_nuxt/" in tag["src"]
]

patterns = [
    r"downloadSelected",
    r"getAll",
    r"fileList",
    r"fileUrl",
    r"downloadFile",
    r"download",
    r"dataSetId",
    r"entityFile",
    r"file/list",
    r"file/download",
]

print("JavaScript bundles found:", len(scripts))
print("=" * 65)

for script_url in scripts:
    try:
        r = session.get(script_url, timeout=20)
        if not r.ok:
            print("Skipped:", script_url, "HTTP", r.status_code)
            continue

        js = r.text
        matches = []

        for pattern in patterns:
            for match in list(
                re.finditer(pattern, js, flags=re.IGNORECASE)
            )[:2]:
                start = max(0, match.start() - 160)
                end = min(len(js), match.end() + 260)
                snippet = re.sub(r"\s+", " ", js[start:end])
                matches.append((pattern, snippet[:450]))

        if matches:
            print("\nBUNDLE:", script_url)
            for pattern, snippet in matches[:8]:
                print(f"\n  Match [{pattern}]:")
                print(" ", snippet)

    except Exception as exc:
        print("\nCould not inspect:", script_url)
        print("Reason:", str(exc)[:200])

print("\nInspection complete; no download endpoint was called.")


JavaScript bundles found: 9

BUNDLE: https://www.scidb.cn/_nuxt/9b26054.js

  Match [getAll]:
  crollMaxWith=748,this.fileTreeClass="data_file_list overflow-y-auto",e("html").css("overflow-y","auto"))},getZone:function(){this.moment_is=c.a.tz.guess()},handGetAll:function(){var e=this,t=e.$utils.isNotEmpty(e.dataSetIdProp)?e.dataSetIdProp:e.$route.query.dataSetId,r=e.$utils.isNotEmpty(e.dataSetTypeProp)?e.dataSetTypeProp:e.$route.query.dataSetType,n=e.$utils.isNotEmpty(e.myVersion)?e.myVersion:e.$route.query.version;e

  Match [getAll]:
  etType,n=e.$utils.isNotEmpty(e.myVersion)?e.myVersion:e.$route.query.version;e.$utils.isEmpty(n)&&(n="V1");var l="https://www.scidb.cn/api/sdb-filetree-service/getAllUrl?dataSetId="+t+"&type="+r+"&version="+n+"&global="+e.moment_is.split("/")[1];if(e.$utils.isNotEmpty(this.$store.state.login.userName)&&(l=l+"&username="+this.$store.state.login.userName),e.$utils.isNotEmpty(this.$route.params.surl)){var c=this.$route.params

  Match [fileList]:
  rn Obj

In [8]:


import requests
import json

DATASET_ID = "76673646fb7241f58aada7b9f24b25fe"

endpoint = "https://www.scidb.cn/api/sdb-filetree-service/getAllUrl"

params = {
    "dataSetId": DATASET_ID,
    "version": "V1",
    "global": "en",
}

print("Requesting dataset file listing...")
print("=" * 65)

try:
    response = requests.get(
        endpoint,
        params=params,
        timeout=30,
        headers={
            "User-Agent": "Mozilla/5.0",
            "Referer": (
                "https://www.scidb.cn/en/detail"
                f"?dataSetId={DATASET_ID}"
            ),
        },
    )

    print("HTTP status:", response.status_code)
    print("Resolved endpoint:", response.url)
    print("Content type:", response.headers.get("Content-Type"))

    print("\nResponse preview:")
    print(response.text[:10000])

    try:
        payload = response.json()

        output_path = (
            "/content/hybrid_gnn_llm/data/raw/"
            "sciencedb_file_listing.json"
        )

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)

        print("\nJSON response saved to:", output_path)

    except ValueError:
        print("\nResponse was not valid JSON; nothing was saved.")

except requests.RequestException as exc:
    print("Request failed:", repr(exc))


Requesting dataset file listing...
HTTP status: 200
Resolved endpoint: https://www.scidb.cn/api/sdb-filetree-service/getAllUrl?dataSetId=76673646fb7241f58aada7b9f24b25fe&version=V1&global=en
Content type: application/octet-stream;charset=UTF-8

Response preview:
https://download.scidb.cn/download?fileId=adadfc1bf205bedf0f068b0dbe33348c&path=/V1/Yelp-Dataset.zip&fileName=Yelp-Dataset.zip


Response was not valid JSON; nothing was saved.


In [9]:

# Cell 9: Download and verify the official ScienceDB archive

import requests
import hashlib
import zipfile
from pathlib import Path

RAW_DIR = Path("/content/hybrid_gnn_llm/data/raw")
YELP_DIR = RAW_DIR / "yelpchi"
YELP_DIR.mkdir(parents=True, exist_ok=True)

archive_url = (
    "https://download.scidb.cn/download"
    "?fileId=adadfc1bf205bedf0f068b0dbe33348c"
    "&path=/V1/Yelp-Dataset.zip"
    "&fileName=Yelp-Dataset.zip"
)

archive_path = YELP_DIR / "Yelp-Dataset.zip"

print("Downloading the archive from ScienceDB...")
print("Destination:", archive_path)

with requests.get(
    archive_url,
    stream=True,
    timeout=(30, 180),
    headers={"User-Agent": "Mozilla/5.0"},
) as response:
    print("HTTP status:", response.status_code)
    response.raise_for_status()

    content_type = response.headers.get("Content-Type", "")
    print("Content type:", content_type)

    downloaded = 0

    with open(archive_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)

            if downloaded and downloaded % (25 * 1024 * 1024) < 1024 * 1024:
                print(f"Downloaded: {downloaded / (1024**2):.1f} MB")

print("\nDownload finished.")
print(f"Archive size: {archive_path.stat().st_size / (1024**2):.2f} MB")

# Compute a checksum so this exact downloaded file can be identified.
sha256 = hashlib.sha256()

with open(archive_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

print("SHA-256:", sha256.hexdigest())

# Inspect archive contents before extraction.
print("\nZIP integrity check...")
with zipfile.ZipFile(archive_path, "r") as zf:
    bad_member = zf.testzip()
    print("Integrity result:", "PASS" if bad_member is None else bad_member)

    print("\nArchive file listing (first 60 entries):")
    for info in zf.infolist()[:60]:
        print(
            f"{info.file_size / (1024**2):8.2f} MB  "
            f"{info.filename}"
        )

    print("\nTotal archive entries:", len(zf.infolist()))


Destination: /content/hybrid_gnn_llm/data/raw/yelpchi/Yelp-Dataset.zip
HTTP status: 200
Content type: application/octet-stream
Downloaded: 25.0 MB
Downloaded: 50.0 MB
Downloaded: 75.0 MB
Downloaded: 100.0 MB
Downloaded: 125.0 MB
Downloaded: 150.0 MB
Downloaded: 175.0 MB
Downloaded: 200.0 MB
Downloaded: 225.0 MB
Downloaded: 250.0 MB

Download finished.
Archive size: 265.29 MB
SHA-256: 65f9ca0fc9f2937a0d419f2c6c771a7eec52d15e5ec96381812b09911b0102a7

ZIP integrity check...
Integrity result: PASS

Archive file listing (first 60 entries):
  111.94 MB  YelpNYC.zip
  198.06 MB  YelpChi.mat
  391.83 MB  yelpzip.csv

Total archive entries: 3


In [10]:

# Cell 10: Check whether the Yelp archive is complete

from pathlib import Path
import zipfile
import os

archive_path = Path(
    "/content/hybrid_gnn_llm/data/raw/yelpchi/Yelp-Dataset.zip"
)

if not archive_path.exists():
    print("Archive file not found. Cell 9 may not have started.")
else:
    size_mb = archive_path.stat().st_size / (1024 ** 2)

    print(f"Current archive size: {size_mb:.2f} MB")
    print(f"File exists: {archive_path.exists()}")

    if size_mb < 200:
        print(
            "\nThe file may still be downloading or may be incomplete."
        )
        print("Do not extract it yet.")
    else:
        print("\nTesting ZIP structure...")

        try:
            with zipfile.ZipFile(archive_path, "r") as zf:
                bad_member = zf.testzip()

                if bad_member is None:
                    print("ZIP integrity: PASS")
                    print("Number of entries:", len(zf.infolist()))

                    print("\nFirst 40 files:")
                    for info in zf.infolist()[:40]:
                        print(
                            f"{info.file_size / (1024**2):.2f} MB  "
                            f"{info.filename}"
                        )
                else:
                    print("ZIP integrity: FAIL")
                    print("Problematic member:", bad_member)

        except zipfile.BadZipFile:
            print("ZIP is incomplete or invalid. Do not extract it.")


Current archive size: 265.29 MB
File exists: True

Testing ZIP structure...
ZIP integrity: PASS
Number of entries: 3

First 40 files:
111.94 MB  YelpNYC.zip
198.06 MB  YelpChi.mat
391.83 MB  yelpzip.csv


In [11]:

# Cell 11: Extract only the YelpChi MATLAB file

from pathlib import Path
import zipfile
import hashlib

archive_path = Path(
    "/content/hybrid_gnn_llm/data/raw/yelpchi/Yelp-Dataset.zip"
)

yelp_dir = Path(
    "/content/hybrid_gnn_llm/data/raw/yelpchi"
)
yelp_dir.mkdir(parents=True, exist_ok=True)

target_name = "YelpChi.mat"
target_path = yelp_dir / target_name

with zipfile.ZipFile(archive_path, "r") as zf:
    matches = [
        name for name in zf.namelist()
        if Path(name).name.lower() == target_name.lower()
    ]

    if not matches:
        print("ERROR: YelpChi.mat was not found in the archive.")
        print("Archive entries:", zf.namelist())
    else:
        member = matches[0]
        print("Selected archive member:", member)

        if not target_path.exists():
            with zf.open(member) as src, open(target_path, "wb") as dst:
                while True:
                    chunk = src.read(1024 * 1024)
                    if not chunk:
                        break
                    dst.write(chunk)

        print("\nExtraction complete.")
        print("File:", target_path)
        print(f"Size: {target_path.stat().st_size / (1024**2):.2f} MB")

        digest = hashlib.sha256()
        with open(target_path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                digest.update(chunk)

        print("SHA-256:", digest.hexdigest())


Selected archive member: YelpChi.mat

Extraction complete.
File: /content/hybrid_gnn_llm/data/raw/yelpchi/YelpChi.mat
Size: 198.06 MB
SHA-256: fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42


In [12]:

# Cell 12: Inspect YelpChi.mat variables

from pathlib import Path
from scipy.io import whosmat

mat_path = Path(
    "/content/hybrid_gnn_llm/data/raw/yelpchi/YelpChi.mat"
)

if not mat_path.exists():
    raise FileNotFoundError(f"Dataset not found: {mat_path}")

print("Inspecting MATLAB file:")
print(mat_path)
print("=" * 70)

variables = whosmat(str(mat_path))

print(f"Number of top-level variables: {len(variables)}\n")

for name, shape, matlab_type in variables:
    print(f"Variable: {name}")
    print(f"  Shape: {shape}")
    print(f"  MATLAB type: {matlab_type}")
    print("-" * 50)


Inspecting MATLAB file:
/content/hybrid_gnn_llm/data/raw/yelpchi/YelpChi.mat
Number of top-level variables: 6

Variable: homo
  Shape: (45954, 45954)
  MATLAB type: sparse
--------------------------------------------------
Variable: net_rur
  Shape: (45954, 45954)
  MATLAB type: sparse
--------------------------------------------------
Variable: net_rtr
  Shape: (45954, 45954)
  MATLAB type: sparse
--------------------------------------------------
Variable: net_rsr
  Shape: (45954, 45954)
  MATLAB type: sparse
--------------------------------------------------
Variable: features
  Shape: (45954, 32)
  MATLAB type: sparse
--------------------------------------------------
Variable: label
  Shape: (1, 45954)
  MATLAB type: int64
--------------------------------------------------


In [13]:

# Cell 13 (fixed): Inspect YelpChi labels, features, and networks

import numpy as np
from scipy.io import loadmat
from scipy import sparse
from collections import Counter
from pathlib import Path

mat_path = Path(
    "/content/hybrid_gnn_llm/data/raw/yelpchi/YelpChi.mat"
)

print("Loading YelpChi variables...")
data = loadmat(str(mat_path))

# 1. Labels
labels_raw = np.asarray(data["label"]).reshape(-1)

print("\n1. LABEL AUDIT")
print("=" * 60)
print("Label vector shape:", labels_raw.shape)
print("Label dtype:", labels_raw.dtype)

for value, count in sorted(Counter(labels_raw.tolist()).items()):
    print(f"  Label {value}: {count:,} ({count / len(labels_raw):.2%})")

# 2. Features
features = data["features"]
features_csr = sparse.csr_matrix(features)

print("\n2. FEATURE MATRIX")
print("=" * 60)
print("Shape:", features_csr.shape)
print("Sparse format:", features.getformat(), "-> CSR for inspection")
print("Stored nonzero values:", f"{features_csr.nnz:,}")
print("Density:", f"{features_csr.nnz / np.prod(features_csr.shape):.6%}")
print("Stored value range:",
      float(features_csr.data.min()),
      "to",
      float(features_csr.data.max()))

print("First 5 rows, first 8 features:")
print(features_csr[:5, :8].toarray())

# 3. Networks
network_names = ["homo", "net_rur", "net_rtr", "net_rsr"]

print("\n3. NETWORK MATRICES")
print("=" * 60)

for name in network_names:
    matrix = sparse.csr_matrix(data[name])

    row_sums = np.asarray(matrix.sum(axis=1)).reshape(-1)
    difference = matrix - matrix.T

    print(f"\n{name}")
    print("  Shape:", matrix.shape)
    print("  Nonzero entries:", f"{matrix.nnz:,}")
    print("  Density:", f"{matrix.nnz / np.prod(matrix.shape):.6%}")
    print("  Isolated rows:", int(np.sum(row_sums == 0)))
    print("  Min row sum:", float(row_sums.min()))
    print("  Max row sum:", float(row_sums.max()))
    print("  Exactly symmetric:", difference.nnz == 0)

    if matrix.nnz:
        print("  Stored value range:",
              float(matrix.data.min()),
              "to",
              float(matrix.data.max()))

print("\nAudit complete. Original labels and matrices were not modified.")


Loading YelpChi variables...

1. LABEL AUDIT
Label vector shape: (45954,)
Label dtype: int64
  Label 0: 39,277 (85.47%)
  Label 1: 6,677 (14.53%)

2. FEATURE MATRIX
Shape: (45954, 32)
Sparse format: coo -> CSR for inspection
Stored nonzero values: 1,469,088
Density: 99.902076%
Stored value range: 1.4837895986330452e-05 to 1.0
First 5 rows, first 8 features:
[[0.02237555 0.07049484 0.42868165 0.99998516 0.99998516 0.39845686
  0.82359225 0.497025  ]
 [0.02492767 0.99998516 0.99998516 0.99998516 0.99998516 0.39845686
  0.99998516 0.94125677]
 [0.00617256 0.07049484 0.42868165 0.99998516 0.99998516 0.39845686
  0.05722976 0.15124267]
 [0.01737518 0.07049484 0.42868165 0.99998516 0.99998516 0.39845686
  0.99998516 0.93020254]
 [0.00908079 0.99998516 0.99998516 0.99998516 0.99998516 0.39845686
  0.15937384 0.69718822]]

3. NETWORK MATRICES

homo
  Shape: (45954, 45954)
  Nonzero entries: 7,693,958
  Density: 0.364337%
  Isolated rows: 13
  Min row sum: 0.0
  Max row sum: 501.0
  Exactly sym

In [14]:

# Cell 14: Graph integrity and dataset consistency checks

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.io import loadmat
from pathlib import Path

mat_path = Path(
    "/content/hybrid_gnn_llm/data/raw/yelpchi/YelpChi.mat"
)
data = loadmat(str(mat_path))

labels = np.asarray(data["label"]).reshape(-1)
X = sparse.csr_matrix(data["features"])

network_names = ["homo", "net_rur", "net_rtr", "net_rsr"]
networks = {
    name: sparse.csr_matrix(data[name])
    for name in network_names
}

print("GRAPH INTEGRITY REPORT")
print("=" * 70)

# 1. Label checks
print("\n1. LABEL CHECKS")
print("Number of nodes:", len(labels))
print("Unique labels:", np.unique(labels).tolist())
print("Missing labels:", int(pd.isna(labels).sum()))
print("Label values preserved; no mapping applied.")

# 2. Feature checks
print("\n2. FEATURE CHECKS")
print("Feature shape:", X.shape)
print("Stored values finite:", bool(np.isfinite(X.data).all()))
print("Feature minimum:", float(X.data.min()))
print("Feature maximum:", float(X.data.max()))
print("Rows with all-zero features:", int(np.sum(X.getnnz(axis=1) == 0)))

# 3. Per-network structural checks
print("\n3. NETWORK CHECKS")

for name, A in networks.items():
    diagonal_nnz = int(A.diagonal().nonzero()[0].size)
    symmetry_diff = A - A.T
    row_degree = np.asarray(A.sum(axis=1)).reshape(-1)

    print(f"\n{name}")
    print("  Nodes:", A.shape[0])
    print("  Nonzero entries:", A.nnz)
    print("  Self-loop entries:", diagonal_nnz)
    print("  Symmetry violations:", symmetry_diff.nnz)
    print("  Degree-zero nodes:", int(np.sum(row_degree == 0)))
    print("  Mean degree:", round(float(row_degree.mean()), 4))
    print("  Median degree:", round(float(np.median(row_degree)), 4))

# 4. Pairwise edge overlap
print("\n4. PAIRWISE EDGE OVERLAP")
print("(Counts use directed matrix entries; symmetric edges appear twice.)")

for i, name_a in enumerate(network_names):
    for name_b in network_names[i + 1:]:
        A = networks[name_a]
        B = networks[name_b]

        overlap = A.multiply(B).nnz
        union = A.nnz + B.nnz - overlap
        jaccard = overlap / union if union else 0.0

        print(
            f"{name_a} vs {name_b}: "
            f"overlap={overlap:,}, Jaccard={jaccard:.6f}"
        )

print("\n5. NODE ALIGNMENT")
expected_shape = (len(labels), len(labels))
for name, A in networks.items():
    print(
        f"{name}: shape matches label vector = "
        f"{A.shape == expected_shape}"
    )

print("\nAudit finished. No source matrices or labels were changed.")


GRAPH INTEGRITY REPORT

1. LABEL CHECKS
Number of nodes: 45954
Unique labels: [0, 1]
Missing labels: 0
Label values preserved; no mapping applied.

2. FEATURE CHECKS
Feature shape: (45954, 32)
Stored values finite: True
Feature minimum: 1.4837895986330452e-05
Feature maximum: 1.0
Rows with all-zero features: 0

3. NETWORK CHECKS

homo
  Nodes: 45954
  Nonzero entries: 7693958
  Self-loop entries: 0
  Symmetry violations: 0
  Degree-zero nodes: 13
  Mean degree: 167.4274
  Median degree: 168.0

net_rur
  Nodes: 45954
  Nonzero entries: 98630
  Self-loop entries: 0
  Symmetry violations: 0
  Degree-zero nodes: 22123
  Mean degree: 2.1463
  Median degree: 1.0

net_rtr
  Nodes: 45954
  Nonzero entries: 1147232
  Self-loop entries: 0
  Symmetry violations: 0
  Degree-zero nodes: 522
  Mean degree: 24.9648
  Median degree: 23.0

net_rsr
  Nodes: 45954
  Nonzero entries: 6805486
  Self-loop entries: 0
  Symmetry violations: 0
  Degree-zero nodes: 40
  Mean degree: 148.0934
  Median degree: 15

In [15]:

# Cell 15: Save dataset and environment audit manifest

import json
import hashlib
import platform
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import scipy
import torch
from scipy import sparse
from scipy.io import loadmat

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
MAT_PATH = PROJECT_DIR / "data/raw/yelpchi/YelpChi.mat"
MANIFEST_PATH = PROJECT_DIR / "results/yelpchi_audit_manifest.json"

# Checksum of the extracted MATLAB file
sha256 = hashlib.sha256()
with open(MAT_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

data = loadmat(str(MAT_PATH))
labels = np.asarray(data["label"]).reshape(-1)
features = sparse.csr_matrix(data["features"])

network_names = ["homo", "net_rur", "net_rtr", "net_rsr"]

manifest = {
    "dataset": "Yelp Dataset / YelpChi.mat",
    "source_doi": "10.57760/sciencedb.17088",
    "extracted_file": str(MAT_PATH),
    "sha256": sha256.hexdigest(),
    "node_count": int(len(labels)),
    "feature_count": int(features.shape[1]),
    "feature_matrix": {
        "shape": list(features.shape),
        "nnz": int(features.nnz),
        "density": float(features.nnz / np.prod(features.shape)),
    },
    "labels": {
        "counts": {
            str(k): int(v)
            for k, v in Counter(labels.tolist()).items()
        },
        "fake_genuine_mapping": "UNVERIFIED; original labels preserved",
    },
    "networks": {},
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "pytorch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    "split_status": "not created",
    "model_status": "not trained",
}

for name in network_names:
    A = sparse.csr_matrix(data[name])
    degrees = np.asarray(A.sum(axis=1)).reshape(-1)

    manifest["networks"][name] = {
        "shape": list(A.shape),
        "nnz": int(A.nnz),
        "density": float(A.nnz / np.prod(A.shape)),
        "isolated_nodes": int(np.sum(degrees == 0)),
        "self_loop_entries": int(A.diagonal().nonzero()[0].size),
        "symmetric": bool((A - A.T).nnz == 0),
    }

MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Manifest saved:", MANIFEST_PATH)
print("Dataset SHA-256:", manifest["sha256"])
print("Nodes:", manifest["node_count"])
print("Features:", manifest["feature_count"])
print("Label counts:", manifest["labels"]["counts"])
print("Networks recorded:", list(manifest["networks"].keys()))
print("CUDA available:", manifest["environment"]["cuda_available"])
print("Model trained:", manifest["model_status"])


Manifest saved: /content/hybrid_gnn_llm/results/yelpchi_audit_manifest.json
Dataset SHA-256: fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42
Nodes: 45954
Features: 32
Label counts: {'0': 39277, '1': 6677}
Networks recorded: ['homo', 'net_rur', 'net_rtr', 'net_rsr']
CUDA available: False
Model trained: not trained


In [16]:

# Cell 16: Create and save stratified splits

import json
import numpy as np
from pathlib import Path
from scipy.io import loadmat
from sklearn.model_selection import train_test_split

SEED = 42

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
MAT_PATH = PROJECT_DIR / "data/raw/yelpchi/YelpChi.mat"
SPLIT_DIR = PROJECT_DIR / "data/processed"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

data = loadmat(str(MAT_PATH))
labels = np.asarray(data["label"]).reshape(-1)

# First split: 70% train, 30% temporary
all_indices = np.arange(len(labels))

train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=SEED,
    stratify=labels,
)

# Split the temporary set equally into validation and test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=SEED,
    stratify=labels[temp_idx],
)

splits = {
    "train": train_idx,
    "validation": val_idx,
    "test": test_idx,
}

print("STRATIFIED SPLIT REPORT")
print("=" * 60)

for name, indices in splits.items():
    subset_labels = labels[indices]
    values, counts = np.unique(subset_labels, return_counts=True)

    print(f"\n{name.upper()}")
    print("  Samples:", len(indices))
    print("  Fraction:", f"{len(indices) / len(labels):.2%}")
    print("  Class counts:", dict(zip(values.tolist(), counts.tolist())))
    print(
        "  Class proportions:",
        {
            int(v): round(float(c / len(indices)), 4)
            for v, c in zip(values, counts)
        },
    )

# Confirm there are no overlapping node indices between splits
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0

# Confirm all nodes are represented exactly once
combined = np.concatenate([train_idx, val_idx, test_idx])
assert len(combined) == len(labels)
assert len(np.unique(combined)) == len(labels)

# Save split indices and their metadata
np.savez_compressed(
    SPLIT_DIR / "yelpchi_splits_seed42.npz",
    train_idx=train_idx,
    validation_idx=val_idx,
    test_idx=test_idx,
)

metadata = {
    "dataset": "YelpChi",
    "seed": SEED,
    "split_ratios": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
    },
    "stratified": True,
    "label_mapping": "unverified; labels preserved as 0 and 1",
    "split_unit": "node",
    "graph_protocol": (
        "Not yet finalized. These are node partitions; "
        "edge visibility must be explicitly specified for training."
    ),
}

with open(
    SPLIT_DIR / "yelpchi_split_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, indent=2)

print("\nSplit overlap checks: PASS")
print("All nodes assigned exactly once: PASS")
print("Saved indices:", SPLIT_DIR / "yelpchi_splits_seed42.npz")
print("Saved metadata:", SPLIT_DIR / "yelpchi_split_metadata.json")


STRATIFIED SPLIT REPORT

TRAIN
  Samples: 32167
  Fraction: 70.00%
  Class counts: {0: 27493, 1: 4674}
  Class proportions: {0: 0.8547, 1: 0.1453}

VALIDATION
  Samples: 6893
  Fraction: 15.00%
  Class counts: {0: 5892, 1: 1001}
  Class proportions: {0: 0.8548, 1: 0.1452}

TEST
  Samples: 6894
  Fraction: 15.00%
  Class counts: {0: 5892, 1: 1002}
  Class proportions: {0: 0.8547, 1: 0.1453}

Split overlap checks: PASS
All nodes assigned exactly once: PASS
Saved indices: /content/hybrid_gnn_llm/data/processed/yelpchi_splits_seed42.npz
Saved metadata: /content/hybrid_gnn_llm/data/processed/yelpchi_split_metadata.json


In [17]:

# Cell 17: Inspect local documentation and MATLAB metadata

from pathlib import Path
from scipy.io import loadmat
import json

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
YELP_DIR = PROJECT_DIR / "data/raw/yelpchi"
MAT_PATH = YELP_DIR / "YelpChi.mat"

print("LOCAL DATASET DOCUMENTATION")
print("=" * 65)

documentation_extensions = {
    ".txt", ".md", ".readme", ".pdf", ".json", ".csv"
}

docs = [
    p for p in YELP_DIR.rglob("*")
    if p.is_file()
    and p != MAT_PATH
    and p.suffix.lower() in documentation_extensions
]

if docs:
    for path in docs:
        print(f"\nFound: {path}")
        print(f"Size: {path.stat().st_size:,} bytes")

        if path.suffix.lower() in {".txt", ".md", ".readme", ".json"}:
            try:
                content = path.read_text(
                    encoding="utf-8",
                    errors="replace"
                )
                print(content[:3000])
            except Exception as exc:
                print("Could not read as text:", exc)
else:
    print("No separate documentation files found.")

print("\nMATLAB FILE METADATA")
print("=" * 65)

data = loadmat(str(MAT_PATH))

for key in ["__header__", "__version__", "__globals__"]:
    value = data.get(key)
    print(f"{key}: {value}")

print("\nVARIABLE NAMES")
print([key for key in data.keys() if not key.startswith("__")])

print("\nCurrent label mapping status:")
print("0 = not independently verified")
print("1 = not independently verified")
print("Relation semantics = not independently verified")

print("\nNo dataset labels or graph relations were modified.")


LOCAL DATASET DOCUMENTATION
No separate documentation files found.

MATLAB FILE METADATA
__header__: b'MATLAB 5.0 MAT-file Platform: posix, Created on: Wed Aug 19 20:09:02 2020'
__version__: 1.0
__globals__: []

VARIABLE NAMES
['homo', 'net_rur', 'net_rtr', 'net_rsr', 'features', 'label']

Current label mapping status:
0 = not independently verified
1 = not independently verified
Relation semantics = not independently verified

No dataset labels or graph relations were modified.


In [18]:

# Cell 18: Prepare label and relation consistency report

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat
from scipy import sparse

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
MAT_PATH = PROJECT_DIR / "data/raw/yelpchi/YelpChi.mat"

data = loadmat(str(MAT_PATH))
labels = np.asarray(data["label"]).reshape(-1)

# Relation names used in the dataset
relation_names = {
    "net_rur": "Review-user-review relation (R-U-R)",
    "net_rtr": "Review-time-review relation (R-T-R)",
    "net_rsr": "Review-star-rating-review relation (R-S-R)",
    "homo": "Homogeneous graph; construction to verify",
}

rows = []

for relation, description in relation_names.items():
    A = sparse.csr_matrix(data[relation])
    degree = np.asarray(A.sum(axis=1)).reshape(-1)

    rows.append({
        "relation": relation,
        "description": description,
        "nodes": A.shape[0],
        "directed_entries": int(A.nnz),
        "undirected_edges": int(A.nnz // 2),
        "isolated_nodes": int(np.sum(degree == 0)),
        "mean_degree": round(float(degree.mean()), 4),
        "symmetric": bool((A - A.T).nnz == 0),
    })

relation_report = pd.DataFrame(rows)

print("LABEL DISTRIBUTION")
print("=" * 65)

label_report = pd.DataFrame({
    "label": np.unique(labels),
    "count": [
        int(np.sum(labels == value))
        for value in np.unique(labels)
    ],
})
label_report["fraction"] = label_report["count"] / len(labels)

print(label_report.to_string(index=False))

print("\nRELATION SUMMARY")
print("=" * 65)
print(relation_report.to_string(index=False))

print("\nINTERPRETATION STATUS")
print("- Labels 0/1: original values preserved; semantic mapping pending.")
print("- R-U-R / R-T-R / R-S-R: published relation definitions noted.")
print("- Exact matrix construction and homo relation: still to verify.")

report_path = PROJECT_DIR / "results" / "yelpchi_relation_report.csv"
relation_report.to_csv(report_path, index=False)

print("\nSaved relation report:", report_path)


LABEL DISTRIBUTION
 label  count  fraction
     0  39277  0.854703
     1   6677  0.145297

RELATION SUMMARY
relation                                description  nodes  directed_entries  undirected_edges  isolated_nodes  mean_degree  symmetric
 net_rur        Review-user-review relation (R-U-R)  45954             98630             49315           22123       2.1463       True
 net_rtr        Review-time-review relation (R-T-R)  45954           1147232            573616             522      24.9648       True
 net_rsr Review-star-rating-review relation (R-S-R)  45954           6805486           3402743              40     148.0934       True
    homo  Homogeneous graph; construction to verify  45954           7693958           3846979              13     167.4274       True

INTERPRETATION STATUS
- Labels 0/1: original values preserved; semantic mapping pending.
- R-U-R / R-T-R / R-S-R: published relation definitions noted.
- Exact matrix construction and homo relation: still to verify.

In [19]:

# Cell 19: Feature-only baseline with class-weighted logistic regression

import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat
from scipy import sparse

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
MAT_PATH = PROJECT_DIR / "data/raw/yelpchi/YelpChi.mat"
SPLIT_PATH = PROJECT_DIR / "data/processed/yelpchi_splits_seed42.npz"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load features and labels
data = loadmat(str(MAT_PATH))
X = sparse.csr_matrix(data["features"])
y = np.asarray(data["label"]).reshape(-1)

# Load the saved, reproducible node split
split = np.load(SPLIT_PATH)
train_idx = split["train_idx"]
val_idx = split["validation_idx"]
test_idx = split["test_idx"]

# Keep original labels 0 and 1; no semantic mapping assumed
model = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42,
        solver="liblinear",
    )),
])

print("Training feature-only logistic regression...")
model.fit(X[train_idx], y[train_idx])

def evaluate_split(name, indices):
    y_true = y[indices]
    y_pred = model.predict(X[indices])
    y_score = model.predict_proba(X[indices])[:, 1]

    metrics = {
        "split": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "f1_class_1": f1_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "roc_auc_class_1": roc_auc_score(y_true, y_score),
        "average_precision_class_1": average_precision_score(
            y_true, y_score
        ),
    }

    print(f"\n{name.upper()} METRICS")
    print("-" * 60)
    for key, value in metrics.items():
        if key != "split":
            print(f"{key}: {value:.4f}")

    print("\nConfusion matrix (rows=true, columns=predicted; labels 0, 1):")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

    print("\nClassification report (labels remain 0 and 1):")
    print(classification_report(
        y_true, y_pred, labels=[0, 1], zero_division=0
    ))

    return metrics

# Validation metrics are for development only.
val_metrics = evaluate_split("validation", val_idx)

# Evaluate test once for this fixed baseline.
# Do not use test results to tune hyperparameters.
test_metrics = evaluate_split("test", test_idx)

metrics_df = pd.DataFrame([val_metrics, test_metrics])
metrics_path = RESULTS_DIR / "yelpchi_logistic_baseline_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print("\nSaved metrics to:", metrics_path)
print("This baseline uses features only; it is not the proposed GNN-LLM model.")


Training feature-only logistic regression...

VALIDATION METRICS
------------------------------------------------------------
accuracy: 0.7101
balanced_accuracy: 0.7073
precision_macro: 0.6133
recall_macro: 0.7073
f1_macro: 0.6105
f1_class_1: 0.4134
roc_auc_class_1: 0.7838
average_precision_class_1: 0.3939

Confusion matrix (rows=true, columns=predicted; labels 0, 1):
[[4191 1701]
 [ 297  704]]

Classification report (labels remain 0 and 1):
              precision    recall  f1-score   support

           0       0.93      0.71      0.81      5892
           1       0.29      0.70      0.41      1001

    accuracy                           0.71      6893
   macro avg       0.61      0.71      0.61      6893
weighted avg       0.84      0.71      0.75      6893


TEST METRICS
------------------------------------------------------------
accuracy: 0.7092
balanced_accuracy: 0.6961
precision_macro: 0.6082
recall_macro: 0.6961
f1_macro: 0.6057
f1_class_1: 0.4038
roc_auc_class_1: 0.7652
aver

In [20]:

# Cell 20: Feature + homo-neighbor aggregation baseline

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import loadmat
from scipy import sparse

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix,
)

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
MAT_PATH = PROJECT_DIR / "data/raw/yelpchi/YelpChi.mat"
SPLIT_PATH = PROJECT_DIR / "data/processed/yelpchi_splits_seed42.npz"
RESULTS_DIR = PROJECT_DIR / "results"

data = loadmat(str(MAT_PATH))
X = sparse.csr_matrix(data["features"], dtype=np.float32)
A = sparse.csr_matrix(data["homo"], dtype=np.float32)

split = np.load(SPLIT_PATH)
train_idx = split["train_idx"]
val_idx = split["validation_idx"]
test_idx = split["test_idx"]

y = np.asarray(data["label"]).reshape(-1)

# Remove self-loops if any; audit previously found none.
A.setdiag(0)
A.eliminate_zeros()

# Symmetric row normalization for neighbor averaging.
degree = np.asarray(A.sum(axis=1)).reshape(-1)
inv_degree = np.zeros_like(degree, dtype=np.float32)
nonzero = degree > 0
inv_degree[nonzero] = 1.0 / degree[nonzero]

D_inv = sparse.diags(inv_degree)
A_mean = D_inv @ A

# Aggregate neighbors' features without densifying the adjacency matrix.
X_neighbors = A_mean @ X

# Combine original and neighborhood features.
X_combined = sparse.hstack(
    [X, X_neighbors],
    format="csr",
    dtype=np.float32,
)

print("Original feature shape:", X.shape)
print("Neighbor feature shape:", X_neighbors.shape)
print("Combined feature shape:", X_combined.shape)
print("Graph:", "homo")
print("Graph protocol: all observed graph edges/features used for aggregation")
print("Label values preserved:", np.unique(y).tolist())

model = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42,
        solver="liblinear",
    )),
])

print("\nTraining feature + neighbor-aggregation baseline...")
model.fit(X_combined[train_idx], y[train_idx])

def evaluate(name, idx):
    truth = y[idx]
    pred = model.predict(X_combined[idx])
    score = model.predict_proba(X_combined[idx])[:, 1]

    result = {
        "split": name,
        "accuracy": accuracy_score(truth, pred),
        "balanced_accuracy": balanced_accuracy_score(truth, pred),
        "precision_macro": precision_score(
            truth, pred, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            truth, pred, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            truth, pred, average="macro", zero_division=0
        ),
        "class_1_f1": f1_score(
            truth, pred, pos_label=1, zero_division=0
        ),
        "roc_auc_class_1": roc_auc_score(truth, score),
        "average_precision_class_1": average_precision_score(truth, score),
    }

    print(f"\n{name.upper()}")
    for key, value in result.items():
        if key != "split":
            print(f"{key}: {value:.4f}")

    print("Confusion matrix:")
    print(confusion_matrix(truth, pred, labels=[0, 1]))
    return result

val_result = evaluate("validation", val_idx)
test_result = evaluate("test", test_idx)

out_path = RESULTS_DIR / "yelpchi_neighbor_baseline_metrics.csv"
pd.DataFrame([val_result, test_result]).to_csv(out_path, index=False)

print("\nSaved metrics:", out_path)
print("This is a neighbor-aggregation baseline, not the GNN-LLM model.")


Original feature shape: (45954, 32)
Neighbor feature shape: (45954, 32)
Combined feature shape: (45954, 64)
Graph: homo
Graph protocol: all observed graph edges/features used for aggregation
Label values preserved: [0, 1]

Training feature + neighbor-aggregation baseline...

VALIDATION
accuracy: 0.7216
balanced_accuracy: 0.7144
precision_macro: 0.6190
recall_macro: 0.7144
macro_f1: 0.6200
class_1_f1: 0.4236
roc_auc_class_1: 0.7932
average_precision_class_1: 0.4092
Confusion matrix:
[[4269 1623]
 [ 296  705]]

TEST
accuracy: 0.7160
balanced_accuracy: 0.7030
precision_macro: 0.6127
recall_macro: 0.7030
macro_f1: 0.6124
class_1_f1: 0.4120
roc_auc_class_1: 0.7732
average_precision_class_1: 0.3928
Confusion matrix:
[[4250 1642]
 [ 316  686]]

Saved metrics: /content/hybrid_gnn_llm/results/yelpchi_neighbor_baseline_metrics.csv
This is a neighbor-aggregation baseline, not the GNN-LLM model.


In [21]:

# Cell 21: Two-layer GCN baseline on YelpChi
# This is a GNN-only baseline, NOT the full GNN-LLM model.

import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from scipy.io import loadmat
from scipy import sparse
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
MAT_PATH = PROJECT_DIR / "data/raw/yelpchi/YelpChi.mat"
SPLIT_PATH = PROJECT_DIR / "data/processed/yelpchi_splits_seed42.npz"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load data
data = loadmat(str(MAT_PATH))
X_sp = sparse.csr_matrix(data["features"], dtype=np.float32)
A_sp = sparse.csr_matrix(data["homo"], dtype=np.float32)
y = np.asarray(data["label"]).reshape(-1).astype(np.int64)

split = np.load(SPLIT_PATH)
train_idx = split["train_idx"]
val_idx = split["validation_idx"]
test_idx = split["test_idx"]

# Symmetrize, remove self-loops, then add identity for GCN normalization.
A_sp = A_sp.maximum(A_sp.T).tocsr()
A_sp.setdiag(0)
A_sp.eliminate_zeros()
A_sp = A_sp + sparse.eye(A_sp.shape[0], dtype=np.float32, format="csr")

# Symmetric normalization: D^(-1/2) (A + I) D^(-1/2)
degree = np.asarray(A_sp.sum(axis=1)).reshape(-1)
degree_inv_sqrt = np.power(np.maximum(degree, 1.0), -0.5)
D_inv_sqrt = sparse.diags(degree_inv_sqrt.astype(np.float32))
A_norm = (D_inv_sqrt @ A_sp @ D_inv_sqrt).tocsr()

# Convert normalized adjacency to a PyTorch sparse tensor.
A_coo = A_norm.tocoo()
indices = torch.tensor(
    np.vstack([A_coo.row, A_coo.col]), dtype=torch.long
)
values = torch.tensor(A_coo.data, dtype=torch.float32)
A_torch = torch.sparse_coo_tensor(
    indices, values, size=A_coo.shape
).coalesce().to(device)

# Dense node features: 45,954 x 32 is manageable.
X = torch.tensor(X_sp.toarray(), dtype=torch.float32, device=device)
y_torch = torch.tensor(y, dtype=torch.long, device=device)

train_t = torch.tensor(train_idx, dtype=torch.long, device=device)
val_t = torch.tensor(val_idx, dtype=torch.long, device=device)
test_t = torch.tensor(test_idx, dtype=torch.long, device=device)

class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, dropout=0.5):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim, bias=False)
        self.fc2 = nn.Linear(hidden_dim, 2, bias=False)
        self.dropout = dropout

    def forward(self, x, adj):
        h = torch.sparse.mm(adj, x)
        h = self.fc1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = torch.sparse.mm(adj, h)
        return self.fc2(h)

model = GCN(in_dim=X.shape[1], hidden_dim=64, dropout=0.5).to(device)

# Inverse-frequency class weights computed from training labels only.
train_labels_np = y[train_idx]
class_counts = np.bincount(train_labels_np, minlength=2)
class_weights = len(train_labels_np) / (
    2.0 * np.maximum(class_counts, 1)
)
class_weights_t = torch.tensor(
    class_weights, dtype=torch.float32, device=device
)

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.01, weight_decay=5e-4
)

def evaluate_gcn(indices_t):
    model.eval()
    with torch.no_grad():
        logits = model(X, A_torch)
        probs = torch.softmax(logits[indices_t], dim=1)[:, 1].cpu().numpy()
        truth = y_torch[indices_t].cpu().numpy()
        pred = (probs >= 0.5).astype(np.int64)

    return {
        "accuracy": accuracy_score(truth, pred),
        "balanced_accuracy": balanced_accuracy_score(truth, pred),
        "precision_macro": precision_score(
            truth, pred, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            truth, pred, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            truth, pred, average="macro", zero_division=0
        ),
        "class_1_f1": f1_score(
            truth, pred, pos_label=1, zero_division=0
        ),
        "roc_auc_class_1": roc_auc_score(truth, probs),
        "average_precision_class_1": average_precision_score(truth, probs),
        "confusion_matrix": confusion_matrix(
            truth, pred, labels=[0, 1]
        ).tolist()
    }

# Select the checkpoint using validation macro-F1.
best_val_f1 = -1.0
best_state = None
best_epoch = 0
patience = 20
epochs_without_improvement = 0
MAX_EPOCHS = 200

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    logits = model(X, A_torch)
    loss = F.cross_entropy(
        logits[train_t], y_torch[train_t],
        weight=class_weights_t
    )
    loss.backward()
    optimizer.step()

    val_metrics = evaluate_gcn(val_t)
    current_f1 = val_metrics["macro_f1"]

    if current_f1 > best_val_f1:
        best_val_f1 = current_f1
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | loss={loss.item():.4f} "
            f"| val_macro_f1={current_f1:.4f}"
        )

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch}.")
        break

# Restore best validation checkpoint; evaluate test only after selection.
model.load_state_dict(best_state)
val_metrics = evaluate_gcn(val_t)
test_metrics = evaluate_gcn(test_t)

print("\nBest epoch:", best_epoch)
for name, metrics in [("VALIDATION", val_metrics), ("TEST", test_metrics)]:
    print(f"\n{name}")
    for key, value in metrics.items():
        print(f"{key}: {value}")

# Save reproducible metrics and training details.
rows = []
for split_name, metrics in [
    ("validation", val_metrics),
    ("test", test_metrics)
]:
    row = {
        "model": "two_layer_gcn_homo",
        "split": split_name,
        "seed": SEED,
        "best_epoch": best_epoch,
        "hidden_dim": 64,
        "dropout": 0.5,
        "learning_rate": 0.01,
        "weight_decay": 5e-4,
        "class_weighting": "inverse_frequency_train_only",
        "graph_protocol": "transductive_all_observed_edges_and_features",
    }
    row.update({
        k: json.dumps(v) if k == "confusion_matrix" else v
        for k, v in metrics.items()
    })
    rows.append(row)

out_path = RESULTS_DIR / "yelpchi_gcn_baseline_metrics.csv"
pd.DataFrame(rows).to_csv(out_path, index=False)

print("\nSaved metrics:", out_path)


Device: cpu


/tmp/ipykernel_689/2747682284.py:65: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  A_torch = torch.sparse_coo_tensor(


Epoch 001 | loss=0.6933 | val_macro_f1=0.1268
Epoch 010 | loss=0.6942 | val_macro_f1=0.1272
Epoch 020 | loss=0.6930 | val_macro_f1=0.4619
Epoch 030 | loss=0.6925 | val_macro_f1=0.4489
Epoch 040 | loss=0.6905 | val_macro_f1=0.5142
Epoch 050 | loss=0.6895 | val_macro_f1=0.5377
Epoch 060 | loss=0.6874 | val_macro_f1=0.3993
Early stopping at epoch 65.

Best epoch: 45

VALIDATION
accuracy: 0.7839837516320906
balanced_accuracy: 0.5419304897410804
precision_macro: 0.546287998322936
recall_macro: 0.5419304897410804
macro_f1: 0.543703643439233
class_1_f1: 0.21258593336858805
roc_auc_class_1: 0.5509426079690846
average_precision_class_1: 0.21009706128385947
confusion_matrix: [[5203, 689], [800, 201]]

TEST
accuracy: 0.7827096025529446
balanced_accuracy: 0.5399089804098525
precision_macro: 0.5440088729765713
recall_macro: 0.5399089804098525
macro_f1: 0.5415677271725657
class_1_f1: 0.20908130939809927
roc_auc_class_1: 0.5659998739791293
average_precision_class_1: 0.22276039757896543
confusion_matr

In [22]:

# Cell 22: GCN threshold diagnosis
# Threshold selection uses validation labels only.

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from scipy.io import loadmat
from scipy import sparse
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
data = loadmat(str(PROJECT_DIR / "data/raw/yelpchi/YelpChi.mat"))
X_sp = sparse.csr_matrix(data["features"], dtype=np.float32)
A_sp = sparse.csr_matrix(data["homo"], dtype=np.float32)
y = np.asarray(data["label"]).reshape(-1).astype(np.int64)

split = np.load(PROJECT_DIR / "data/processed/yelpchi_splits_seed42.npz")
train_idx = split["train_idx"]
val_idx = split["validation_idx"]
test_idx = split["test_idx"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

A_sp = A_sp.maximum(A_sp.T).tocsr()
A_sp.setdiag(0)
A_sp.eliminate_zeros()
A_sp = A_sp + sparse.eye(A_sp.shape[0], dtype=np.float32, format="csr")

degree = np.asarray(A_sp.sum(axis=1)).reshape(-1)
d_inv_sqrt = np.power(np.maximum(degree, 1.0), -0.5)
D = sparse.diags(d_inv_sqrt.astype(np.float32))
A_norm = (D @ A_sp @ D).tocoo()

A_torch = torch.sparse_coo_tensor(
    torch.tensor(np.vstack([A_norm.row, A_norm.col]), dtype=torch.long),
    torch.tensor(A_norm.data, dtype=torch.float32),
    size=A_norm.shape
).coalesce().to(device)

X = torch.tensor(X_sp.toarray(), dtype=torch.float32, device=device)
y_t = torch.tensor(y, dtype=torch.long, device=device)
train_t = torch.tensor(train_idx, dtype=torch.long, device=device)

class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, dropout=0.5):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim, bias=False)
        self.fc2 = nn.Linear(hidden_dim, 2, bias=False)
        self.dropout = dropout

    def forward(self, x, adj):
        h = torch.sparse.mm(adj, x)
        h = F.relu(self.fc1(h))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = torch.sparse.mm(adj, h)
        return self.fc2(h)

model = GCN(X.shape[1]).to(device)

counts = np.bincount(y[train_idx], minlength=2)
weights = len(train_idx) / (2.0 * np.maximum(counts, 1))
weights_t = torch.tensor(weights, dtype=torch.float32, device=device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.01, weight_decay=5e-4
)

best_f1 = -1.0
best_state = None
best_epoch = 0
patience = 20
wait = 0

def probabilities(indices):
    model.eval()
    with torch.no_grad():
        logits = model(X, A_torch)[indices]
        return torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()
    logits = model(X, A_torch)
    loss = F.cross_entropy(
        logits[train_t], y_t[train_t], weight=weights_t
    )
    loss.backward()
    optimizer.step()

    val_probs = probabilities(
        torch.tensor(val_idx, dtype=torch.long, device=device)
    )
    val_pred = (val_probs >= 0.5).astype(int)
    val_f1 = f1_score(
        y[val_idx], val_pred, average="macro", zero_division=0
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
        wait = 0
    else:
        wait += 1

    if wait >= patience:
        break

model.load_state_dict(best_state)
val_probs = probabilities(
    torch.tensor(val_idx, dtype=torch.long, device=device)
)
test_probs = probabilities(
    torch.tensor(test_idx, dtype=torch.long, device=device)
)

# Search thresholds on validation only.
thresholds = np.arange(0.05, 0.951, 0.01)
threshold_rows = []

for threshold in thresholds:
    pred = (val_probs >= threshold).astype(int)
    threshold_rows.append({
        "threshold": float(threshold),
        "macro_f1": f1_score(
            y[val_idx], pred, average="macro", zero_division=0
        ),
        "class_1_f1": f1_score(
            y[val_idx], pred, pos_label=1, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(y[val_idx], pred),
        "accuracy": accuracy_score(y[val_idx], pred),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df = threshold_df.sort_values(
    ["macro_f1", "threshold"], ascending=[False, True]
)
best_threshold = float(threshold_df.iloc[0]["threshold"])

def report(split_name, truth, probs, threshold):
    pred = (probs >= threshold).astype(int)
    result = {
        "split": split_name,
        "threshold": threshold,
        "accuracy": accuracy_score(truth, pred),
        "balanced_accuracy": balanced_accuracy_score(truth, pred),
        "precision_macro": precision_score(
            truth, pred, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            truth, pred, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            truth, pred, average="macro", zero_division=0
        ),
        "class_1_f1": f1_score(
            truth, pred, pos_label=1, zero_division=0
        ),
        "roc_auc_class_1": roc_auc_score(truth, probs),
        "average_precision_class_1": average_precision_score(truth, probs),
        "confusion_matrix": confusion_matrix(
            truth, pred, labels=[0, 1]
        ).tolist()
    }
    print(f"\n{split_name.upper()} at threshold {threshold:.2f}")
    for key, value in result.items():
        if key not in ("split", "threshold"):
            print(f"{key}: {value}")
    return result

print("Best validation-selected threshold:", best_threshold)
print("Best checkpoint epoch:", best_epoch)

val_result = report("validation", y[val_idx], val_probs, best_threshold)
test_result = report("test", y[test_idx], test_probs, best_threshold)

out_dir = PROJECT_DIR / "results"
threshold_df.to_csv(out_dir / "yelpchi_gcn_threshold_sweep.csv", index=False)
pd.DataFrame([val_result, test_result]).to_json(
    out_dir / "yelpchi_gcn_threshold_metrics.json",
    orient="records", indent=2
)
print("\nSaved threshold sweep and selected-threshold metrics.")


Device: cpu
Best validation-selected threshold: 0.5000000000000001
Best checkpoint epoch: 45

VALIDATION at threshold 0.50
accuracy: 0.7839837516320906
balanced_accuracy: 0.5419304897410804
precision_macro: 0.546287998322936
recall_macro: 0.5419304897410804
macro_f1: 0.543703643439233
class_1_f1: 0.21258593336858805
roc_auc_class_1: 0.5509426079690846
average_precision_class_1: 0.21009706128385947
confusion_matrix: [[5203, 689], [800, 201]]

TEST at threshold 0.50
accuracy: 0.7827096025529446
balanced_accuracy: 0.5399089804098525
precision_macro: 0.5440088729765713
recall_macro: 0.5399089804098525
macro_f1: 0.5415677271725657
class_1_f1: 0.20908130939809927
roc_auc_class_1: 0.5659998739791293
average_precision_class_1: 0.22276039757896543
confusion_matrix: [[5198, 694], [804, 198]]

Saved threshold sweep and selected-threshold metrics.


In [23]:

# Cell 23: Audit YelpChi labels, dimensions, and saved experiment files

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from scipy.io import loadmat

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
MAT_PATH = PROJECT_DIR / "data/raw/yelpchi/YelpChi.mat"
RESULTS_DIR = PROJECT_DIR / "results"

# 1. Verify source file and checksum
sha256 = hashlib.sha256()
with open(MAT_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

print("DATASET FILE:", MAT_PATH)
print("SHA256:", sha256.hexdigest())

# 2. Inspect labels and matrix dimensions
data = loadmat(str(MAT_PATH))
labels = np.asarray(data["label"]).reshape(-1)
features = data["features"]

print("\nDATASET DIMENSIONS")
print("Number of nodes / labels:", len(labels))
print("Feature matrix shape:", features.shape)
print("Label values and counts:")
values, counts = np.unique(labels, return_counts=True)
for value, count in zip(values, counts):
    print(f"  label {value}: {count} ({count / len(labels):.2%})")

print("\nMAT FILE VARIABLES:")
for key, value in data.items():
    if not key.startswith("__"):
        print(f"  {key}: shape={getattr(value, 'shape', None)}")

# 3. Compare against manuscript's reported total
paper_total = 67395
print("\nMANUSCRIPT CONSISTENCY CHECK")
print("Manuscript-reported YelpChi total:", paper_total)
print("Downloaded YelpChi total:", len(labels))
print("Difference:", paper_total - len(labels))
print("Counts match:", len(labels) == paper_total)

# 4. Confirm experiment artifacts exist
print("\nSAVED EXPERIMENT FILES")
for path in sorted(RESULTS_DIR.glob("*")):
    if path.is_file():
        print(f"  {path.name}: {path.stat().st_size:,} bytes")

# 5. Inspect audit metadata, if available
manifest_path = RESULTS_DIR / "yelpchi_audit_manifest.json"
if manifest_path.exists():
    print("\nEXISTING AUDIT MANIFEST:")
    print(manifest_path.read_text()[:6000])
else:
    print("\nNo audit manifest found at:", manifest_path)

print(
    "\nIMPORTANT: This cell verifies label values and file consistency; "
    "it does not establish whether label 0 or label 1 means fake reviews."
)


DATASET FILE: /content/hybrid_gnn_llm/data/raw/yelpchi/YelpChi.mat
SHA256: fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42

DATASET DIMENSIONS
Number of nodes / labels: 45954
Feature matrix shape: (45954, 32)
Label values and counts:
  label 0: 39277 (85.47%)
  label 1: 6677 (14.53%)

MAT FILE VARIABLES:
  homo: shape=(45954, 45954)
  net_rur: shape=(45954, 45954)
  net_rtr: shape=(45954, 45954)
  net_rsr: shape=(45954, 45954)
  features: shape=(45954, 32)
  label: shape=(1, 45954)

MANUSCRIPT CONSISTENCY CHECK
Manuscript-reported YelpChi total: 67395
Downloaded YelpChi total: 45954
Difference: 21441
Counts match: False

SAVED EXPERIMENT FILES
  yelpchi_audit_manifest.json: 1,808 bytes
  yelpchi_gcn_baseline_metrics.csv: 854 bytes
  yelpchi_gcn_threshold_metrics.json: 891 bytes
  yelpchi_gcn_threshold_sweep.csv: 7,133 bytes
  yelpchi_logistic_baseline_metrics.csv: 446 bytes
  yelpchi_neighbor_baseline_metrics.csv: 442 bytes
  yelpchi_relation_report.csv: 436 bytes

EXI

In [24]:

# Cell 24: Find local dataset documentation and metadata

from pathlib import Path
import zipfile

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
RAW_DIR = PROJECT_DIR / "data/raw/yelpchi"

print("PROJECT:", PROJECT_DIR)
print("\nPotential documentation files:")

extensions = {".txt", ".md", ".pdf", ".json", ".csv", ".html", ".xml"}
found = []

for base in [PROJECT_DIR / "data", PROJECT_DIR / "results"]:
    if base.exists():
        for path in base.rglob("*"):
            if path.is_file() and (
                path.suffix.lower() in extensions
                and path.name.lower() not in {
                    "yelpchi_relation_report.csv",
                    "yelpchi_logistic_baseline_metrics.csv",
                    "yelpchi_neighbor_baseline_metrics.csv",
                    "yelpchi_gcn_baseline_metrics.csv",
                    "yelpchi_gcn_threshold_sweep.csv",
                }
            ):
                found.append(path)

for path in sorted(set(found)):
    print(f"  {path.relative_to(PROJECT_DIR)} ({path.stat().st_size:,} bytes)")

zip_path = RAW_DIR / "Yelp-Dataset.zip"
if zip_path.exists():
    print("\nArchive contents that may contain documentation:")
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            name = info.filename
            if any(word in name.lower() for word in
                   ["readme", "description", "label", "yelpchi", "metadata"]):
                print(f"  {name} ({info.file_size:,} bytes)")
else:
    print("\nOriginal ZIP not found at:", zip_path)

print("\nAudit manifest:")
manifest = PROJECT_DIR / "results/yelpchi_audit_manifest.json"
if manifest.exists():
    print(manifest.read_text()[:3000])


PROJECT: /content/hybrid_gnn_llm

Potential documentation files:
  data/processed/yelpchi_split_metadata.json (360 bytes)
  results/yelpchi_audit_manifest.json (1,808 bytes)
  results/yelpchi_gcn_threshold_metrics.json (891 bytes)

Archive contents that may contain documentation:
  YelpChi.mat (207,676,376 bytes)

Audit manifest:
{
  "dataset": "Yelp Dataset / YelpChi.mat",
  "source_doi": "10.57760/sciencedb.17088",
  "extracted_file": "/content/hybrid_gnn_llm/data/raw/yelpchi/YelpChi.mat",
  "sha256": "fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42",
  "node_count": 45954,
  "feature_count": 32,
  "feature_matrix": {
    "shape": [
      45954,
      32
    ],
    "nnz": 1469088,
    "density": 0.9990207598903251
  },
  "labels": {
    "counts": {
      "0": 39277,
      "1": 6677
    },
    "fake_genuine_mapping": "UNVERIFIED; original labels preserved"
  },
  "networks": {
    "homo": {
      "shape": [
        45954,
        45954
      ],
      "nnz": 7693958,
 

In [25]:

# Cell 25: Record verified dataset semantics in the audit manifest

from pathlib import Path
import json

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
manifest_path = PROJECT_DIR / "results/yelpchi_audit_manifest.json"

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

manifest["labels"]["fake_genuine_mapping"] = (
    "label 1 = spam/fraudulent; label 0 = benign, "
    "based on public YelpChi graph dataset documentation"
)

manifest["network_relations"] = {
    "net_rur": "R-U-R: reviews posted by the same user",
    "net_rtr": (
        "R-T-R: reviews for the same product posted in the same month"
    ),
    "net_rsr": (
        "R-S-R: reviews for the same product with the same star rating"
    ),
    "homo": (
        "Homogeneous aggregate graph; exact construction in this file "
        "has not been independently established"
    ),
}

manifest["dataset_size_discrepancy"] = {
    "downloaded_graph_nodes": 45954,
    "manuscript_reported_nodes": 67395,
    "difference": 21441,
    "status": (
        "UNRESOLVED: likely different dataset representation or "
        "preprocessing; do not claim exact dataset-size reproduction"
    ),
}

manifest["split_status"] = (
    "Stratified 70/15/15 node split, seed 42; "
    "split files saved separately"
)
manifest["model_status"] = (
    "Feature-only logistic, neighbor-aggregation logistic, and "
    "two-layer GCN baselines trained; hybrid GNN-LLM not yet reproduced"
)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Updated:", manifest_path)
print(json.dumps({
    "labels": manifest["labels"],
    "network_relations": manifest["network_relations"],
    "dataset_size_discrepancy": manifest["dataset_size_discrepancy"],
    "split_status": manifest["split_status"],
    "model_status": manifest["model_status"],
}, indent=2))


Updated: /content/hybrid_gnn_llm/results/yelpchi_audit_manifest.json
{
  "labels": {
    "counts": {
      "0": 39277,
      "1": 6677
    },
    "fake_genuine_mapping": "label 1 = spam/fraudulent; label 0 = benign, based on public YelpChi graph dataset documentation"
  },
  "network_relations": {
    "net_rur": "R-U-R: reviews posted by the same user",
    "net_rtr": "R-T-R: reviews for the same product posted in the same month",
    "net_rsr": "R-S-R: reviews for the same product with the same star rating",
    "homo": "Homogeneous aggregate graph; exact construction in this file has not been independently established"
  },
  "dataset_size_discrepancy": {
    "downloaded_graph_nodes": 45954,
    "manuscript_reported_nodes": 67395,
    "difference": 21441,
    "status": "UNRESOLVED: likely different dataset representation or preprocessing; do not claim exact dataset-size reproduction"
  },
  "split_status": "Stratified 70/15/15 node split, seed 42; split files saved separately",
  "mo

In [26]:

# Cell 26: Relation-aware GNN baseline on YelpChi
# Three relation-specific graph convolutions, followed by feature fusion.
# This is a GNN baseline, NOT the full GNN-LLM model.

import random
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from scipy.io import loadmat
from scipy import sparse
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data = loadmat(str(PROJECT_DIR / "data/raw/yelpchi/YelpChi.mat"))
X_sp = sparse.csr_matrix(data["features"], dtype=np.float32)
y = np.asarray(data["label"]).reshape(-1).astype(np.int64)

split = np.load(
    PROJECT_DIR / "data/processed/yelpchi_splits_seed42.npz"
)
train_idx = split["train_idx"]
val_idx = split["validation_idx"]
test_idx = split["test_idx"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Nodes:", X_sp.shape[0], "| Features:", X_sp.shape[1])

# Normalize each relation independently; add self-loops.
relation_names = ["net_rur", "net_rtr", "net_rsr"]
adjacencies = []

for relation_name in relation_names:
    A = sparse.csr_matrix(data[relation_name], dtype=np.float32)
    A = A.maximum(A.T).tocsr()
    A.setdiag(0)
    A.eliminate_zeros()
    A = A + sparse.eye(A.shape[0], dtype=np.float32, format="csr")

    degree = np.asarray(A.sum(axis=1)).reshape(-1)
    inv_sqrt = np.power(np.maximum(degree, 1.0), -0.5)
    D = sparse.diags(inv_sqrt.astype(np.float32))
    A_norm = (D @ A @ D).tocoo()

    A_t = torch.sparse_coo_tensor(
        torch.tensor(np.vstack([A_norm.row, A_norm.col]), dtype=torch.long),
        torch.tensor(A_norm.data, dtype=torch.float32),
        size=A_norm.shape
    ).coalesce().to(device)

    adjacencies.append(A_t)
    print(
        relation_name,
        "| nonzero normalized entries:", A_norm.nnz
    )

X = torch.tensor(X_sp.toarray(), dtype=torch.float32, device=device)
y_t = torch.tensor(y, dtype=torch.long, device=device)
train_t = torch.tensor(train_idx, dtype=torch.long, device=device)
val_t = torch.tensor(val_idx, dtype=torch.long, device=device)
test_t = torch.tensor(test_idx, dtype=torch.long, device=device)

class RelationGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, dropout=0.5):
        super().__init__()
        self.self_layer = nn.Linear(in_dim, hidden_dim)
        self.rel_layers = nn.ModuleList([
            nn.Linear(in_dim, hidden_dim, bias=False)
            for _ in range(3)
        ])
        self.fusion = nn.Linear(hidden_dim * 4, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, x, adjs):
        own = self.self_layer(x)
        messages = [torch.sparse.mm(adj, x) for adj in adjs]
        relation_messages = [
            layer(message)
            for layer, message in zip(self.rel_layers, messages)
        ]
        h = torch.cat([own] + relation_messages, dim=1)
        h = F.relu(self.fusion(h))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.classifier(h)

model = RelationGCN(X.shape[1]).to(device)

# Class weights estimated only from training labels.
counts = np.bincount(y[train_idx], minlength=2)
weights = len(train_idx) / (2.0 * np.maximum(counts, 1))
weights_t = torch.tensor(weights, dtype=torch.float32, device=device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.01, weight_decay=5e-4
)

def evaluate(indices_t):
    model.eval()
    with torch.no_grad():
        logits = model(X, adjacencies)[indices_t]
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        truth = y_t[indices_t].cpu().numpy()
        pred = (probs >= 0.5).astype(int)

    return {
        "accuracy": accuracy_score(truth, pred),
        "balanced_accuracy": balanced_accuracy_score(truth, pred),
        "precision_macro": precision_score(
            truth, pred, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            truth, pred, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            truth, pred, average="macro", zero_division=0
        ),
        "class_1_f1": f1_score(
            truth, pred, pos_label=1, zero_division=0
        ),
        "roc_auc_class_1": roc_auc_score(truth, probs),
        "average_precision_class_1": average_precision_score(truth, probs),
        "confusion_matrix": confusion_matrix(
            truth, pred, labels=[0, 1]
        ).tolist()
    }

best_val_f1 = -1.0
best_state = None
best_epoch = 0
patience = 20
wait = 0
MAX_EPOCHS = 150

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits = model(X, adjacencies)
    loss = F.cross_entropy(
        logits[train_t], y_t[train_t], weight=weights_t
    )
    loss.backward()
    optimizer.step()

    val_metrics = evaluate(val_t)
    current_f1 = val_metrics["macro_f1"]

    if current_f1 > best_val_f1:
        best_val_f1 = current_f1
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
        wait = 0
    else:
        wait += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | loss={loss.item():.4f} "
            f"| val_macro_f1={current_f1:.4f}"
        )

    if wait >= patience:
        print("Early stopping at epoch", epoch)
        break

model.load_state_dict(best_state)
val_metrics = evaluate(val_t)
test_metrics = evaluate(test_t)

print("\nBest epoch:", best_epoch)
for name, metrics in [
    ("VALIDATION", val_metrics),
    ("TEST", test_metrics)
]:
    print(f"\n{name}")
    for key, value in metrics.items():
        print(f"{key}: {value}")

rows = []
for split_name, metrics in [
    ("validation", val_metrics),
    ("test", test_metrics)
]:
    row = {
        "model": "three_relation_gcn",
        "split": split_name,
        "seed": SEED,
        "best_epoch": best_epoch,
        "hidden_dim": 64,
        "dropout": 0.5,
        "learning_rate": 0.01,
        "weight_decay": 5e-4,
        "class_weighting": "inverse_frequency_train_only",
        "relations": ",".join(relation_names),
        "graph_protocol": "transductive_all_observed_edges_and_features",
    }
    row.update({
        k: json.dumps(v) if k == "confusion_matrix" else v
        for k, v in metrics.items()
    })
    rows.append(row)

out_path = RESULTS_DIR / "yelpchi_relation_gcn_metrics.csv"
pd.DataFrame(rows).to_csv(out_path, index=False)
print("\nSaved metrics:", out_path)


Device: cpu
Nodes: 45954 | Features: 32
net_rur | nonzero normalized entries: 144584
net_rtr | nonzero normalized entries: 1193186
net_rsr | nonzero normalized entries: 6851440
Epoch 001 | loss=0.7019 | val_macro_f1=0.4609
Epoch 010 | loss=0.6764 | val_macro_f1=0.1285
Epoch 020 | loss=0.6136 | val_macro_f1=0.5308
Epoch 030 | loss=0.5781 | val_macro_f1=0.6080
Epoch 040 | loss=0.5599 | val_macro_f1=0.6338
Epoch 050 | loss=0.5487 | val_macro_f1=0.6427
Early stopping at epoch 55

Best epoch: 35

VALIDATION
accuracy: 0.8092267517771652
balanced_accuracy: 0.7192350249885892
precision_macro: 0.6598398541318994
recall_macro: 0.7192350249885892
macro_f1: 0.6788419767228244
class_1_f1: 0.47421031587365053
roc_auc_class_1: 0.8006371598530458
average_precision_class_1: 0.46351238621788093
confusion_matrix: [[4985, 907], [408, 593]]

TEST
accuracy: 0.8035973310124747
balanced_accuracy: 0.7037046070791209
precision_macro: 0.6500898571271678
recall_macro: 0.7037046070791209
macro_f1: 0.66735639003086

In [27]:

# Cell 27: Consolidate test metrics from all baseline models

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
RESULTS_DIR = PROJECT_DIR / "results"

files = {
    "Feature-only logistic regression":
        "yelpchi_logistic_baseline_metrics.csv",
    "Neighbor-aggregation logistic regression":
        "yelpchi_neighbor_baseline_metrics.csv",
    "Single-graph GCN":
        "yelpchi_gcn_baseline_metrics.csv",
    "Three-relation GNN":
        "yelpchi_relation_gcn_metrics.csv",
}

required_metrics = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "class_1_f1",
    "roc_auc_class_1",
    "average_precision_class_1",
]

rows = []

for model_name, filename in files.items():
    path = RESULTS_DIR / filename

    if not path.exists():
        print("MISSING:", filename)
        continue

    df = pd.read_csv(path)

    if "split" in df.columns:
        test_rows = df[df["split"].astype(str).str.lower() == "test"]
        if test_rows.empty:
            print("No test row found in:", filename)
            continue
        row = test_rows.iloc[0]
    else:
        # Older baseline files may contain validation/test as separate rows
        # or a metric column; inspect their schema instead of guessing.
        print("Unexpected schema in:", filename, list(df.columns))
        continue

    result = {"model": model_name}
    for metric in required_metrics:
        if metric in df.columns:
            result[metric] = float(row[metric])
        else:
            result[metric] = None

    rows.append(result)

comparison = pd.DataFrame(rows)

if not comparison.empty:
    comparison = comparison.sort_values(
        "macro_f1", ascending=False, na_position="last"
    )
    output_path = RESULTS_DIR / "yelpchi_baseline_comparison.csv"
    comparison.to_csv(output_path, index=False)

    print("\nBASELINE COMPARISON")
    print(comparison.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("\nSaved:", output_path)
else:
    print(
        "\nNo comparison file was generated. Share the schema messages "
        "above so we can adapt the consolidation cell."
    )



BASELINE COMPARISON
                                   model  accuracy  balanced_accuracy  macro_f1  class_1_f1  roc_auc_class_1  average_precision_class_1
                      Three-relation GNN    0.8036             0.7037    0.6674      0.4545           0.7834                     0.4449
Neighbor-aggregation logistic regression    0.7160             0.7030    0.6124      0.4120           0.7732                     0.3928
                        Single-graph GCN    0.7827             0.5399    0.5416      0.2091           0.5660                     0.2228
        Feature-only logistic regression    0.7092             0.6961       NaN         NaN           0.7652                     0.3712

Saved: /content/hybrid_gnn_llm/results/yelpchi_baseline_comparison.csv


In [28]:

# Cell 28: Fix the consolidated baseline comparison

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
RESULTS_DIR = PROJECT_DIR / "results"

comparison_path = RESULTS_DIR / "yelpchi_baseline_comparison.csv"
comparison = pd.read_csv(comparison_path)

print("Current comparison columns:", comparison.columns.tolist())

# Known feature-only test metrics from the completed baseline run.
# These are the previously observed test results, not new measurements.
feature_only_test = {
    "accuracy": 0.7092,
    "balanced_accuracy": 0.6961,
    "macro_f1": 0.6057,
    "class_1_f1": 0.4038,
    "roc_auc_class_1": 0.7652,
    "average_precision_class_1": 0.3712,
}

mask = comparison["model"].eq("Feature-only logistic regression")
if mask.any():
    for metric, value in feature_only_test.items():
        if metric in comparison.columns and comparison.loc[mask, metric].isna().any():
            comparison.loc[mask, metric] = value

# Round only for display/storage; preserve four decimal places.
metric_columns = [
    "accuracy", "balanced_accuracy", "macro_f1",
    "class_1_f1", "roc_auc_class_1",
    "average_precision_class_1"
]
for column in metric_columns:
    if column in comparison.columns:
        comparison[column] = pd.to_numeric(
            comparison[column], errors="coerce"
        ).round(4)

comparison = comparison.sort_values(
    "macro_f1", ascending=False, na_position="last"
).reset_index(drop=True)

comparison.to_csv(comparison_path, index=False)

print("\nCorrected baseline comparison:")
print(comparison.to_string(index=False))
print("\nSaved:", comparison_path)

# Report missing metrics, if any.
print("\nRemaining missing values:")
print(comparison[["model"] + metric_columns].isna().sum())


Current comparison columns: ['model', 'accuracy', 'balanced_accuracy', 'macro_f1', 'class_1_f1', 'roc_auc_class_1', 'average_precision_class_1']

Corrected baseline comparison:
                                   model  accuracy  balanced_accuracy  macro_f1  class_1_f1  roc_auc_class_1  average_precision_class_1
                      Three-relation GNN    0.8036             0.7037    0.6674      0.4545           0.7834                     0.4449
Neighbor-aggregation logistic regression    0.7160             0.7030    0.6124      0.4120           0.7732                     0.3928
        Feature-only logistic regression    0.7092             0.6961    0.6057      0.4038           0.7652                     0.3712
                        Single-graph GCN    0.7827             0.5399    0.5416      0.2091           0.5660                     0.2228

Saved: /content/hybrid_gnn_llm/results/yelpchi_baseline_comparison.csv

Remaining missing values:
model                        0
accuracy     

In [29]:

from pathlib import Path
import shutil
import zipfile

project = Path("/content/hybrid_gnn_llm")
backup = Path("/content/hybrid_gnn_llm_backup.zip")

if not project.exists():
    raise FileNotFoundError(f"Project folder not found: {project}")

# Include the project files and saved results.
with zipfile.ZipFile(
    backup, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for path in project.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(project.parent))

print("Backup created:", backup)
print(f"ZIP size: {backup.stat().st_size / (1024**2):.2f} MB")
print("Contains saved baseline metrics and audit files.")


Backup created: /content/hybrid_gnn_llm_backup.zip
ZIP size: 280.49 MB
Contains saved baseline metrics and audit files.


In [30]:

from google.colab import files
files.download("/content/hybrid_gnn_llm_backup.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:

from pathlib import Path
import zipfile

PROJECT_DIR = Path("/content/hybrid_gnn_llm")

print("=== PROJECT DATA FILES ===")

for folder_name in ["data/raw", "data/processed", "cache"]:
    folder = PROJECT_DIR / folder_name
    print(f"\n[{folder_name}]")

    if not folder.exists():
        print("Folder not found")
        continue

    files = [
        p for p in folder.rglob("*")
        if p.is_file()
    ]

    if not files:
        print("No files found")
        continue

    for p in files[:30]:
        print(f"{p.relative_to(PROJECT_DIR)} ({p.stat().st_size / 1e6:.2f} MB)")

    if len(files) > 30:
        print(f"... and {len(files) - 30} more files")

print("\n=== POSSIBLE REVIEW-TEXT FILES ===")

text_extensions = {
    ".csv", ".json", ".jsonl", ".txt",
    ".parquet", ".tsv", ".feather"
}

candidates = [
    p for folder in [
        PROJECT_DIR / "data/raw",
        PROJECT_DIR / "data/processed"
    ]
    if folder.exists()
    for p in folder.rglob("*")
    if p.is_file() and p.suffix.lower() in text_extensions
]

if candidates:
    for p in candidates:
        print(p.relative_to(PROJECT_DIR))
else:
    print("No obvious review-text files found.")


=== PROJECT DATA FILES ===

[data/raw]
data/raw/yelpchi/Yelp-Dataset.zip (278.17 MB)
data/raw/yelpchi/YelpChi.mat (207.68 MB)

[data/processed]
data/processed/yelpchi_splits_seed42.npz (0.13 MB)
data/processed/yelpchi_split_metadata.json (0.00 MB)

[cache]
No files found

=== POSSIBLE REVIEW-TEXT FILES ===
data/processed/yelpchi_split_metadata.json


In [32]:

from pathlib import Path
import zipfile

zip_path = Path(
    "/content/hybrid_gnn_llm/data/raw/yelpchi/Yelp-Dataset.zip"
)

print("Archive exists:", zip_path.exists())

if zip_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = zf.namelist()

        print("Number of archive entries:", len(members))
        print("\nFirst 40 entries:")

        for name in members[:40]:
            info = zf.getinfo(name)
            print(f"{name} ({info.file_size / 1e6:.2f} MB)")

        print("\nPotential text or metadata files:")
        candidates = [
            name for name in members
            if name.lower().endswith((
                ".csv", ".json", ".jsonl", ".txt",
                ".tsv", ".parquet", ".xlsx"
            ))
        ]

        if candidates:
            for name in candidates[:100]:
                print(name)
        else:
            print("No obvious text/metadata files found.")


Archive exists: True
Number of archive entries: 3

First 40 entries:
YelpNYC.zip (117.37 MB)
YelpChi.mat (207.68 MB)
yelpzip.csv (410.87 MB)

Potential text or metadata files:
yelpzip.csv


In [33]:

from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_DIR = Path("/content/hybrid_gnn_llm")
PROCESSED_DIR = PROJECT_DIR / "data/processed"
RESULTS_DIR = PROJECT_DIR / "results"

split_path = PROCESSED_DIR / "yelpchi_splits_seed42.npz"
graph_path = PROJECT_DIR / "data/raw/yelpchi/YelpChi.mat"

assert split_path.exists(), f"Missing split file: {split_path}"
assert graph_path.exists(), f"Missing graph file: {graph_path}"

splits = np.load(split_path)

required_splits = {
    "train_idx", "validation_idx", "test_idx"
}
assert required_splits.issubset(splits.files), (
    f"Expected {required_splits}; found {splits.files}"
)

train_idx = splits["train_idx"].astype(np.int64)
val_idx = splits["validation_idx"].astype(np.int64)
test_idx = splits["test_idx"].astype(np.int64)

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0

print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")
print("PyTorch:", torch.__version__)
print("Training nodes:", len(train_idx))
print("Validation nodes:", len(val_idx))
print("Test nodes:", len(test_idx))
print("Split integrity: PASS")
print("Ready for model implementation.")


Device: CPU
PyTorch: 2.11.0+cpu
Training nodes: 32167
Validation nodes: 6893
Test nodes: 6894
Split integrity: PASS
Ready for model implementation.


In [34]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class HybridGNNLLM(nn.Module):
    """
    Architecture scaffold for the manuscript's hybrid model.

    Inputs:
      x: node features [num_nodes, input_dim]
      adj: normalized sparse adjacency matrix
      semantic: semantic embeddings [num_nodes, semantic_dim]

    Note:
      This uses a simple sparse graph-convolution implementation.
      It is not yet the manuscript's exact heterogeneous HeteroConv model.
    """

    def __init__(
        self,
        input_dim,
        semantic_dim=128,
        hidden_dim=128,
        num_classes=2,
        dropout=0.5
    ):
        super().__init__()

        self.gnn1 = nn.Linear(input_dim, hidden_dim)
        self.gnn2 = nn.Linear(hidden_dim, hidden_dim)

        self.semantic_projection = nn.Linear(
            semantic_dim, hidden_dim
        )

        self.fusion_gate = nn.Linear(
            hidden_dim * 2, 1
        )

        self.classifier = nn.Linear(
            hidden_dim, num_classes
        )

        self.dropout = dropout

    def forward(self, x, adj, semantic):
        # Structural representation: two graph-convolution steps
        h = torch.sparse.mm(adj, x)
        h = F.relu(self.gnn1(h))
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = torch.sparse.mm(adj, h)
        h = F.relu(self.gnn2(h))

        # Project semantic embeddings to the same dimension
        s = F.relu(self.semantic_projection(semantic))

        # Adaptive structural-semantic fusion
        alpha = torch.sigmoid(
            self.fusion_gate(torch.cat([h, s], dim=1))
        )

        z = alpha * h + (1.0 - alpha) * s
        logits = self.classifier(z)

        return logits, h, s, alpha


print("Hybrid architecture class defined successfully.")
print("Structural hidden dimension: 128")
print("Semantic hidden dimension: 128")
print("Fusion: learned sigmoid gate")
print("Classification output: 2 classes")


Hybrid architecture class defined successfully.
Structural hidden dimension: 128
Semantic hidden dimension: 128
Fusion: learned sigmoid gate
Classification output: 2 classes


In [35]:

from pathlib import Path
import numpy as np
import scipy.io as sio
import scipy.sparse as sp

mat_path = Path(
    "/content/hybrid_gnn_llm/data/raw/yelpchi/YelpChi.mat"
)
assert mat_path.exists(), f"Dataset not found: {mat_path}"

data = sio.loadmat(mat_path)

required = ["features", "label", "net_rur", "net_rtr", "net_rsr"]
missing = [key for key in required if key not in data]
assert not missing, f"Missing expected variables: {missing}"

features = data["features"]
labels = np.asarray(data["label"]).reshape(-1).astype(np.int64)

if not sp.issparse(features):
    features = sp.csr_matrix(features)
else:
    features = features.tocsr()

relations = {
    "net_rur": data["net_rur"],
    "net_rtr": data["net_rtr"],
    "net_rsr": data["net_rsr"],
}

for name, adj in relations.items():
    if not sp.issparse(adj):
        adj = sp.csr_matrix(adj)
    else:
        adj = adj.tocsr()
    adj.eliminate_zeros()
    relations[name] = adj

print("Feature matrix:", features.shape)
print("Labels:", labels.shape)
print("Label values/counts:", dict(zip(*np.unique(labels, return_counts=True))))

for name, adj in relations.items():
    print(
        f"{name}: shape={adj.shape}, "
        f"nonzero_entries={adj.nnz}"
    )

assert features.shape[0] == len(labels)
assert all(adj.shape == (len(labels), len(labels))
           for adj in relations.values())

print("\nFeature/label/graph dimensions are consistent.")


Feature matrix: (45954, 32)
Labels: (45954,)
Label values/counts: {np.int64(0): np.int64(39277), np.int64(1): np.int64(6677)}
net_rur: shape=(45954, 45954), nonzero_entries=98630
net_rtr: shape=(45954, 45954), nonzero_entries=1147232
net_rsr: shape=(45954, 45954), nonzero_entries=6805486

Feature/label/graph dimensions are consistent.


In [36]:

import scipy.sparse as sp
import numpy as np

def normalize_adjacency(adj):
    """Compute D^(-1/2) (A + I) D^(-1/2), preserving sparsity."""
    adj = adj.tocsr().astype(np.float32)
    adj = adj.maximum(adj.T)

    # Add self-loops
    adj = adj + sp.eye(adj.shape[0], dtype=np.float32, format="csr")
    adj.sum_duplicates()
    adj.eliminate_zeros()

    degree = np.asarray(adj.sum(axis=1)).ravel()
    inv_sqrt_degree = np.zeros_like(degree, dtype=np.float32)

    nonzero = degree > 0
    inv_sqrt_degree[nonzero] = degree[nonzero] ** -0.5

    D = sp.diags(inv_sqrt_degree)
    normalized = (D @ adj @ D).tocsr().astype(np.float32)

    return normalized

normalized_relations = {
    name: normalize_adjacency(adj)
    for name, adj in relations.items()
}

for name, adj in normalized_relations.items():
    print(
        f"{name}: shape={adj.shape}, "
        f"nonzeros={adj.nnz}, "
        f"dtype={adj.dtype}, "
        f"finite={np.isfinite(adj.data).all()}"
    )

assert all(
    adj.shape == (len(labels), len(labels))
    and np.isfinite(adj.data).all()
    for adj in normalized_relations.values()
)

print("\nThree normalized relation matrices prepared successfully.")


net_rur: shape=(45954, 45954), nonzeros=144584, dtype=float32, finite=True
net_rtr: shape=(45954, 45954), nonzeros=1193186, dtype=float32, finite=True
net_rsr: shape=(45954, 45954), nonzeros=6851440, dtype=float32, finite=True

Three normalized relation matrices prepared successfully.


In [37]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
import numpy as np

def scipy_to_torch_sparse(matrix):
    matrix = matrix.tocoo().astype(np.float32)
    indices = torch.from_numpy(
        np.vstack([matrix.row, matrix.col]).astype(np.int64)
    )
    values = torch.from_numpy(matrix.data)
    return torch.sparse_coo_tensor(
        indices, values, size=matrix.shape
    ).coalesce()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

adj_tensors = {
    name: scipy_to_torch_sparse(adj).to(device)
    for name, adj in normalized_relations.items()
}

x = torch.from_numpy(features.toarray().astype(np.float32)).to(device)
y = torch.from_numpy(labels).long().to(device)

class ThreeRelationGNN(nn.Module):
    def __init__(self, input_dim=32, hidden_dim=128, dropout=0.5):
        super().__init__()

        self.relation_layers_1 = nn.ModuleDict({
            name: nn.Linear(input_dim, hidden_dim)
            for name in ["net_rur", "net_rtr", "net_rsr"]
        })

        self.relation_layers_2 = nn.ModuleDict({
            name: nn.Linear(hidden_dim, hidden_dim)
            for name in ["net_rur", "net_rtr", "net_rsr"]
        })

        self.fusion_1 = nn.Linear(hidden_dim * 3, hidden_dim)
        self.fusion_2 = nn.Linear(hidden_dim * 3, hidden_dim)
        self.dropout = dropout

    def forward(self, x, adjacencies):
        first_layer = []

        for name, adj in adjacencies.items():
            messages = torch.sparse.mm(adj, x)
            first_layer.append(
                F.relu(self.relation_layers_1[name](messages))
            )

        h = F.relu(self.fusion_1(torch.cat(first_layer, dim=1)))
        h = F.dropout(h, p=self.dropout, training=self.training)

        second_layer = []

        for name, adj in adjacencies.items():
            messages = torch.sparse.mm(adj, h)
            second_layer.append(
                F.relu(self.relation_layers_2[name](messages))
            )

        h = self.fusion_2(torch.cat(second_layer, dim=1))
        return h

structural_encoder = ThreeRelationGNN().to(device)

print("Device:", device)
print("Feature tensor:", tuple(x.shape))
print("Label tensor:", tuple(y.shape))
print("Relation tensors:", list(adj_tensors.keys()))
print("Structural encoder parameters:",
      sum(p.numel() for p in structural_encoder.parameters()))
print("Three-relation GNN encoder defined successfully.")


Device: cpu
Feature tensor: (45954, 32)
Label tensor: (45954,)
Relation tensors: ['net_rur', 'net_rtr', 'net_rsr']
Structural encoder parameters: 160768
Three-relation GNN encoder defined successfully.


In [38]:

structural_encoder.eval()

with torch.no_grad():
    structural_embeddings = structural_encoder(x, adj_tensors)

print("Structural embedding shape:", tuple(structural_embeddings.shape))
print("All values finite:", torch.isfinite(structural_embeddings).all().item())
print("Embedding mean:", structural_embeddings.mean().item())
print("Embedding standard deviation:", structural_embeddings.std().item())

assert structural_embeddings.shape == (len(labels), 128)
assert torch.isfinite(structural_embeddings).all()

print("\nForward-pass validation: PASS")


Structural embedding shape: (45954, 128)
All values finite: True
Embedding mean: -0.0028220824897289276
Embedding standard deviation: 0.04308144375681877

Forward-pass validation: PASS


In [39]:

import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, roc_auc_score,
    average_precision_score, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

split = np.load(
    "/content/hybrid_gnn_llm/data/processed/yelpchi_splits_seed42.npz"
)
train_idx = torch.as_tensor(split["train_idx"], dtype=torch.long, device=device)
val_idx = torch.as_tensor(split["validation_idx"], dtype=torch.long, device=device)
test_idx = torch.as_tensor(split["test_idx"], dtype=torch.long, device=device)

# Classification head on top of the three-relation structural encoder.
class StructuralClassifier(nn.Module):
    def __init__(self, encoder, hidden_dim=128, dropout=0.5):
        super().__init__()
        self.encoder = encoder
        self.dropout = dropout
        self.classifier = nn.Linear(hidden_dim, 2)

    def forward(self, features, relations):
        h = self.encoder(features, relations)
        return self.classifier(F.dropout(h, p=self.dropout, training=self.training))

model = StructuralClassifier(structural_encoder).to(device)

# Calculate class weights from TRAINING labels only.
train_counts = torch.bincount(y[train_idx], minlength=2).float()
class_weights = (train_counts.sum() / (2.0 * train_counts)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(
    model.parameters(), lr=0.001, weight_decay=5e-4
)

def evaluate_split(model, idx):
    model.eval()
    with torch.no_grad():
        logits = model(x, adj_tensors)[idx]
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        true = y[idx].cpu().numpy()
        pred = (probs >= 0.5).astype(np.int64)

    return {
        "accuracy": accuracy_score(true, pred),
        "balanced_accuracy": balanced_accuracy_score(true, pred),
        "macro_precision": precision_score(
            true, pred, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            true, pred, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            true, pred, average="macro", zero_division=0
        ),
        "class1_f1": f1_score(true, pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(true, probs),
        "average_precision": average_precision_score(true, probs),
        "confusion_matrix": confusion_matrix(true, pred).tolist()
    }

best_val_f1 = -1.0
best_epoch = 0
best_state = None
patience = 10
epochs_without_improvement = 0
MAX_EPOCHS = 100

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    logits = model(x, adj_tensors)
    loss = criterion(logits[train_idx], y[train_idx])
    loss.backward()
    optimizer.step()

    val_metrics = evaluate_split(model, val_idx)
    val_f1 = val_metrics["macro_f1"]

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:03d} | loss={loss.item():.4f} "
            f"| val_macro_F1={val_f1:.4f} "
            f"| best={best_val_f1:.4f} (epoch {best_epoch})"
        )

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch}.")
        break

model.load_state_dict(best_state)
val_results = evaluate_split(model, val_idx)
test_results = evaluate_split(model, test_idx)

print("\nBest epoch:", best_epoch)
print("Validation metrics:", val_results)
print("Test metrics:", test_results)


Epoch 001 | loss=0.6937 | val_macro_F1=0.1268 | best=0.1268 (epoch 1)
Epoch 005 | loss=0.6878 | val_macro_F1=0.4669 | best=0.4789 (epoch 2)
Epoch 010 | loss=0.6686 | val_macro_F1=0.4609 | best=0.6451 (epoch 6)
Epoch 015 | loss=0.6259 | val_macro_F1=0.4649 | best=0.6451 (epoch 6)
Early stopping at epoch 16.

Best epoch: 6
Validation metrics: {'accuracy': 0.770636877992166, 'balanced_accuracy': np.float64(0.7020523604026659), 'macro_precision': 0.6303669229772874, 'macro_recall': 0.7020523604026659, 'macro_f1': 0.6450616020369568, 'class1_f1': 0.4339419978517723, 'roc_auc': np.float64(0.7680160131789461), 'average_precision': np.float64(0.39328995415866846), 'confusion_matrix': [[4706, 1186], [395, 606]]}
Test metrics: {'accuracy': 0.7667536988685814, 'balanced_accuracy': np.float64(0.6974731799130862), 'macro_precision': 0.6267838096248943, 'macro_recall': 0.6974731799130862, 'macro_f1': 0.6406416197000926, 'class1_f1': 0.4277580071174377, 'roc_auc': np.float64(0.760218276955932), 'aver

In [40]:

import os
import pandas as pd
import zipfile

project_dir = "/content/hybrid_gnn_llm"
raw_dir = os.path.join(project_dir, "data", "raw", "yelpchi")

csv_path = os.path.join(raw_dir, "yelpzip.csv")

if os.path.exists(csv_path):
    print("CSV path:", csv_path)
    print("File size (MB):", round(os.path.getsize(csv_path) / 1e6, 2))

    df_preview = pd.read_csv(csv_path, nrows=5, low_memory=False)
    print("\nColumns:")
    print(df_preview.columns.tolist())

    print("\nSample rows:")
    display(df_preview.head())

    print("\nData types:")
    print(df_preview.dtypes)
else:
    print("yelpzip.csv not found at:", csv_path)
    print("Files in raw directory:")
    for root, dirs, files in os.walk(raw_dir):
        for name in files:
            print(os.path.join(root, name))


yelpzip.csv not found at: /content/hybrid_gnn_llm/data/raw/yelpchi/yelpzip.csv
Files in raw directory:
/content/hybrid_gnn_llm/data/raw/yelpchi/Yelp-Dataset.zip
/content/hybrid_gnn_llm/data/raw/yelpchi/YelpChi.mat


In [41]:

import os
import zipfile
import pandas as pd

raw_dir = "/content/hybrid_gnn_llm/data/raw/yelpchi"
archive_path = os.path.join(raw_dir, "Yelp-Dataset.zip")
csv_path = os.path.join(raw_dir, "yelpzip.csv")

with zipfile.ZipFile(archive_path, "r") as z:
    matches = [
        name for name in z.namelist()
        if os.path.basename(name).lower() == "yelpzip.csv"
    ]
    print("Matching archive entries:", matches)

    if not matches:
        raise FileNotFoundError("yelpzip.csv was not found inside the archive.")

    # Extract just the matching CSV; avoid extracting the other large datasets.
    with z.open(matches[0]) as src, open(csv_path, "wb") as dst:
        import shutil
        shutil.copyfileobj(src, dst)

print("\nExtracted file size (MB):", round(os.path.getsize(csv_path) / 1e6, 2))

preview = pd.read_csv(csv_path, nrows=5, low_memory=False)
print("\nColumn names:", preview.columns.tolist())
print("\nSample rows:")
display(preview.head())
print("\nData types:")
print(preview.dtypes)


Matching archive entries: ['yelpzip.csv']

Extracted file size (MB): 410.87

Column names: ['Unnamed: 0', 'user_id', 'prod_id', 'rating', 'label', 'date', 'text', 'tag']

Sample rows:


,Unnamed: 0,user_id,prod_id,rating,label,date,text,tag
0,0,5044,0,1.0,-1,2014-11-16,"Drinks were bad, the hot chocolate was watered...",fake
1,1,5045,0,1.0,-1,2014-09-08,This was the worst experience I've ever had a ...,fake
2,2,5046,0,3.0,-1,2013-10-06,This is located on the site of the old Spruce ...,fake
3,3,5047,0,5.0,-1,2014-11-30,I enjoyed coffee and breakfast twice at Toast ...,fake
4,4,5048,0,5.0,-1,2014-08-28,I love Toast! The food choices are fantastic -...,fake



Data types:
Unnamed: 0      int64
user_id         int64
prod_id         int64
rating        float64
label           int64
date           object
text           object
tag            object
dtype: object


In [42]:

import pandas as pd
import numpy as np
import os

csv_path = "/content/hybrid_gnn_llm/data/raw/yelpchi/yelpzip.csv"

# Load only the columns needed for the audit.
reviews = pd.read_csv(
    csv_path,
    usecols=["user_id", "prod_id", "rating", "label", "date", "text", "tag"],
    low_memory=False
)

print("Rows:", len(reviews))
print("Columns:", reviews.columns.tolist())
print("\nMissing values:")
print(reviews[["user_id", "prod_id", "rating", "label", "date", "text", "tag"]].isna().sum())

print("\nLabel distribution:")
print(reviews["label"].value_counts(dropna=False).sort_index())

print("\nTag distribution:")
print(reviews["tag"].value_counts(dropna=False))

print("\nText-length statistics:")
text_lengths = reviews["text"].fillna("").astype(str).str.len()
print(text_lengths.describe(percentiles=[0.25, 0.5, 0.75, 0.95]))

print("\nUnique users:", reviews["user_id"].nunique())
print("Unique products:", reviews["prod_id"].nunique())
print("Duplicate user-product pairs:", reviews.duplicated(["user_id", "prod_id"]).sum())

print("\nExample text and labels:")
display(reviews[["rating", "label", "tag", "text"]].sample(
    n=min(5, len(reviews)), random_state=42
))


Rows: 608458
Columns: ['user_id', 'prod_id', 'rating', 'label', 'date', 'text', 'tag']

Missing values:
user_id    0
prod_id    0
rating     0
label      0
date       0
text       0
tag        0
dtype: int64

Label distribution:
label
-1     80439
 1    528019
Name: count, dtype: int64

Tag distribution:
tag
real    528019
fake     80439
Name: count, dtype: int64

Text-length statistics:
count    608458.000000
mean        629.131690
std         579.854507
min           1.000000
25%         243.000000
50%         462.000000
75%         825.000000
95%        1737.000000
max       29063.000000
Name: text, dtype: float64

Unique users: 260239
Unique products: 5044
Duplicate user-product pairs: 0

Example text and labels:


,rating,label,tag,text
193079,1.0,-1,fake,Lunch with friends on June 12. Two of us order...
102557,3.0,1,real,Ok - first of all this is not real Korean Food...
591602,5.0,1,real,I came here for my birthday with 11 people. Th...
515544,4.0,1,real,What a great place. The food was so good. We ...
116192,5.0,1,real,Just go... seriously... just go! AMAZING. I wa...


In [43]:

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

csv_path = "/content/hybrid_gnn_llm/data/raw/yelpchi/yelpzip.csv"
out_dir = "/content/hybrid_gnn_llm/data/processed"
os.makedirs(out_dir, exist_ok=True)

reviews = pd.read_csv(
    csv_path,
    usecols=["user_id", "prod_id", "rating", "date", "text", "label", "tag"],
    low_memory=False
)

reviews["text"] = reviews["text"].astype(str).str.strip()
reviews = reviews[reviews["text"].str.len() > 0].copy()

# Explicit mapping: fake -> 1, real -> 0.
reviews["target"] = reviews["tag"].map({"fake": 1, "real": 0})

# Check that the independent label column agrees with tag.
expected_label = reviews["tag"].map({"fake": -1, "real": 1})
assert reviews["target"].notna().all(), "Unexpected tag value found."
assert (reviews["label"].to_numpy() == expected_label.to_numpy()).all(), \
    "The label and tag columns disagree."

reviews["target"] = reviews["target"].astype("int64")

all_idx = np.arange(len(reviews))
train_idx, temp_idx = train_test_split(
    all_idx, test_size=0.30, random_state=42,
    stratify=reviews["target"].to_numpy()
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=42,
    stratify=reviews["target"].to_numpy()[temp_idx]
)

np.savez_compressed(
    os.path.join(out_dir, "yelpzip_text_splits_seed42.npz"),
    train_idx=train_idx, validation_idx=val_idx, test_idx=test_idx
)

reviews[["user_id", "prod_id", "rating", "date", "text", "target"]].to_pickle(
    os.path.join(out_dir, "yelpzip_text_clean.pkl")
)

print("Clean reviews:", len(reviews))
print("Label mapping verified: fake=1, real=0")
print("Train / validation / test:", len(train_idx), len(val_idx), len(test_idx))
print("\nTraining distribution:")
print(reviews.iloc[train_idx]["target"].value_counts().sort_index())
print("\nValidation distribution:")
print(reviews.iloc[val_idx]["target"].value_counts().sort_index())
print("\nTest distribution:")
print(reviews.iloc[test_idx]["target"].value_counts().sort_index())
print("\nSaved clean data and splits.")


Clean reviews: 608458
Label mapping verified: fake=1, real=0
Train / validation / test: 425920 91269 91269

Training distribution:
target
0    369613
1     56307
Name: count, dtype: int64

Validation distribution:
target
0    79203
1    12066
Name: count, dtype: int64

Test distribution:
target
0    79203
1    12066
Name: count, dtype: int64

Saved clean data and splits.


In [44]:

%pip install -q sentence-transformers


In [45]:

import os
import pandas as pd

project_dir = "/content/hybrid_gnn_llm"
processed_dir = os.path.join(project_dir, "data", "processed")

reviews = pd.read_pickle(
    os.path.join(processed_dir, "yelpzip_text_clean.pkl")
)

# Exact-size stratified sample, preserving the source class proportions.
subset = (
    reviews.groupby("target", group_keys=False)
    .sample(frac=50000 / len(reviews), random_state=42)
)

# Correct any rounding difference while preserving the class proportions.
if len(subset) > 50000:
    subset = subset.sample(n=50000, random_state=42)
elif len(subset) < 50000:
    extra = reviews.drop(index=subset.index).sample(
        n=50000 - len(subset), random_state=42
    )
    subset = pd.concat([subset, extra])

subset = subset.sample(frac=1, random_state=42).reset_index(drop=True)

subset_path = os.path.join(processed_dir, "yelpzip_dev_subset.pkl")
subset.to_pickle(subset_path)

print("Subset shape:", subset.shape)
print("\nClass counts (0=real, 1=fake):")
print(subset["target"].value_counts().sort_index())
print("\nClass proportions:")
print(subset["target"].value_counts(normalize=True).sort_index())
print("Saved:", subset_path)

assert len(subset) == 50000
assert subset["text"].notna().all()
assert subset["target"].isin([0, 1]).all()
print("\nValidation: PASS")


Subset shape: (50000, 6)

Class counts (0=real, 1=fake):
target
0    43390
1     6610
Name: count, dtype: int64

Class proportions:
target
0    0.8678
1    0.1322
Name: proportion, dtype: float64
Saved: /content/hybrid_gnn_llm/data/processed/yelpzip_dev_subset.pkl

Validation: PASS


In [46]:

import os
import pandas as pd

project_dir = "/content/hybrid_gnn_llm"
processed_dir = os.path.join(project_dir, "data", "processed")

reviews = pd.read_pickle(
    os.path.join(processed_dir, "yelpzip_text_clean.pkl")
)

subset = (
    reviews.groupby("target", group_keys=False)
    .sample(frac=10000 / len(reviews), random_state=42)
)

# Ensure exactly 10,000 rows despite rounding.
if len(subset) > 10000:
    subset = subset.sample(n=10000, random_state=42)
elif len(subset) < 10000:
    extra = reviews.drop(index=subset.index).sample(
        n=10000 - len(subset), random_state=42
    )
    subset = pd.concat([subset, extra])

subset = subset.sample(frac=1, random_state=42).reset_index(drop=True)

subset_path = os.path.join(processed_dir, "yelpzip_dev_10k.pkl")
subset.to_pickle(subset_path)

print("Subset shape:", subset.shape)
print("\nLabels (0=real, 1=fake):")
print(subset["target"].value_counts().sort_index())
print("Saved:", subset_path)

assert len(subset) == 10000
assert subset["text"].notna().all()
assert subset["target"].isin([0, 1]).all()
print("Validation: PASS")


Subset shape: (10000, 6)

Labels (0=real, 1=fake):
target
0    8678
1    1322
Name: count, dtype: int64
Saved: /content/hybrid_gnn_llm/data/processed/yelpzip_dev_10k.pkl
Validation: PASS


In [47]:

import os
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

project_dir = "/content/hybrid_gnn_llm"
processed_dir = os.path.join(project_dir, "data", "processed")
cache_dir = os.path.join(project_dir, "cache")
os.makedirs(cache_dir, exist_ok=True)

subset = pd.read_pickle(
    os.path.join(processed_dir, "yelpzip_dev_10k.pkl")
).reset_index(drop=True)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_path = os.path.join(cache_dir, "yelpzip_dev_10k_embeddings.npy")
progress_path = os.path.join(cache_dir, "yelpzip_dev_10k_progress.json")

encoder = SentenceTransformer(model_name, device="cpu")
texts = subset["text"].astype(str).tolist()
total = len(texts)
dim = encoder.get_sentence_embedding_dimension()
checkpoint_size = 1000
completed = 0

# Resume only from a checkpoint belonging to this exact 10k run.
if os.path.exists(progress_path) and os.path.exists(embedding_path):
    with open(progress_path) as f:
        progress = json.load(f)
    if (progress.get("total") == total
        and progress.get("model_name") == model_name):
        completed = int(progress.get("completed", 0))
        print(f"Resuming from {completed:,}/{total:,}")

if completed == 0:
    embeddings = np.lib.format.open_memmap(
        embedding_path, mode="w+", dtype=np.float32,
        shape=(total, dim)
    )
else:
    embeddings = np.lib.format.open_memmap(
        embedding_path, mode="r+", dtype=np.float32,
        shape=(total, dim)
    )

for start in range(completed, total, checkpoint_size):
    end = min(start + checkpoint_size, total)
    batch_embeddings = encoder.encode(
        texts[start:end],
        batch_size=64,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    embeddings[start:end] = batch_embeddings
    embeddings.flush()

    with open(progress_path, "w") as f:
        json.dump({
            "total": total,
            "model_name": model_name,
            "completed": end
        }, f)

    print(f"Checkpoint saved: {end:,}/{total:,}")

print("\nEmbedding shape:", embeddings.shape)
print("All values finite:", bool(np.isfinite(embeddings).all()))
assert embeddings.shape == (50000, 384)
assert np.isfinite(embeddings).all()
print("Semantic embeddings: PASS")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_689/2641596337.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = encoder.get_sentence_embedding_dimension()


Checkpoint saved: 1,000/10,000
Checkpoint saved: 2,000/10,000
Checkpoint saved: 3,000/10,000
Checkpoint saved: 4,000/10,000
Checkpoint saved: 5,000/10,000
Checkpoint saved: 6,000/10,000
Checkpoint saved: 7,000/10,000
Checkpoint saved: 8,000/10,000
Checkpoint saved: 9,000/10,000
Checkpoint saved: 10,000/10,000

Embedding shape: (10000, 384)
All values finite: True
Semantic embeddings: PASS


In [48]:

import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.model_selection import train_test_split

project_dir = "/content/hybrid_gnn_llm"
processed_dir = os.path.join(project_dir, "data", "processed")
cache_dir = os.path.join(project_dir, "cache")

reviews = pd.read_pickle(
    os.path.join(processed_dir, "yelpzip_dev_10k.pkl")
).reset_index(drop=True)

embeddings = np.load(
    os.path.join(cache_dir, "yelpzip_dev_10k_embeddings.npy"),
    mmap_mode="r"
)

assert len(reviews) == embeddings.shape[0] == 10000

n = len(reviews)
rows = np.arange(n)

def shared_entity_adjacency(entity_ids, n_nodes):
    """Connect reviews sharing a user or product, without self-loops."""
    codes, _ = pd.factorize(entity_ids, sort=False)
    groups = pd.Series(rows).groupby(codes, sort=False).apply(list)

    edge_rows, edge_cols = [], []
    for nodes in groups:
        if len(nodes) < 2:
            continue
        # Avoid quadratic cliques for unusually large groups.
        # Connect each review to up to 20 other reviews in its group.
        nodes = np.asarray(nodes, dtype=np.int64)
        for i, node in enumerate(nodes):
            neighbors = np.delete(nodes, i)
            if len(neighbors) > 20:
                neighbors = neighbors[:20]
            edge_rows.extend([node] * len(neighbors))
            edge_cols.extend(neighbors.tolist())

    adj = sp.coo_matrix(
        (np.ones(len(edge_rows), dtype=np.float32),
         (edge_rows, edge_cols)),
        shape=(n_nodes, n_nodes)
    ).tocsr()

    # Symmetrize and binarize.
    adj = adj.maximum(adj.T)
    adj.setdiag(0)
    adj.eliminate_zeros()
    adj.data[:] = 1.0
    return adj

adj_user = shared_entity_adjacency(reviews["user_id"].to_numpy(), n)
adj_product = shared_entity_adjacency(reviews["prod_id"].to_numpy(), n)

# Add self-loops and symmetric degree normalization.
def normalize_adjacency(adj):
    adj = adj.astype(np.float32).tocsr()
    adj = adj + sp.eye(adj.shape[0], dtype=np.float32, format="csr")
    degree = np.asarray(adj.sum(axis=1)).ravel()
    inv_sqrt = np.power(np.maximum(degree, 1.0), -0.5)
    d = sp.diags(inv_sqrt)
    return (d @ adj @ d).tocsr()

normalized_user = normalize_adjacency(adj_user)
normalized_product = normalize_adjacency(adj_product)

# Reproducible stratified split for these 10,000 review nodes.
labels = reviews["target"].to_numpy(dtype=np.int64)
all_idx = np.arange(n)

train_idx, temp_idx = train_test_split(
    all_idx, test_size=0.30, random_state=42, stratify=labels
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=42,
    stratify=labels[temp_idx]
)

split_path = os.path.join(processed_dir, "yelpzip_dev_10k_splits_seed42.npz")
np.savez_compressed(
    split_path,
    train_idx=train_idx,
    validation_idx=val_idx,
    test_idx=test_idx
)

print("Nodes:", n)
print("Semantic embedding shape:", embeddings.shape)
print("User-relation edges (undirected):", adj_user.nnz // 2)
print("Product-relation edges (undirected):", adj_product.nnz // 2)
print("Train / validation / test:", len(train_idx), len(val_idx), len(test_idx))
print("Saved splits:", split_path)

assert normalized_user.shape == (n, n)
assert normalized_product.shape == (n, n)
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
print("Graph and split validation: PASS")


Nodes: 10000
Semantic embedding shape: (10000, 384)
User-relation edges (undirected): 860
Product-relation edges (undirected): 53125
Train / validation / test: 7000 1500 1500
Saved splits: /content/hybrid_gnn_llm/data/processed/yelpzip_dev_10k_splits_seed42.npz
Graph and split validation: PASS


In [49]:

import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
project_dir = "/content/hybrid_gnn_llm"
processed_dir = os.path.join(project_dir, "data", "processed")
cache_dir = os.path.join(project_dir, "cache")

reviews = pd.read_pickle(
    os.path.join(processed_dir, "yelpzip_dev_10k.pkl")
).reset_index(drop=True)

semantic_np = np.load(
    os.path.join(cache_dir, "yelpzip_dev_10k_embeddings.npy")
).astype(np.float32)

def load_normalized_graph(filename):
    # Rebuild graph consistently from the exact saved review ordering.
    return None

# Reconstruct the two graphs using the same logic as Cell 50.
def make_relation(entity_ids, n):
    codes, _ = pd.factorize(entity_ids, sort=False)
    groups = pd.Series(np.arange(n)).groupby(codes, sort=False).apply(list)
    rr, cc = [], []
    for nodes in groups:
        if len(nodes) < 2:
            continue
        nodes = np.asarray(nodes, dtype=np.int64)
        for i, node in enumerate(nodes):
            neighbors = np.delete(nodes, i)
            if len(neighbors) > 20:
                neighbors = neighbors[:20]
            rr.extend([node] * len(neighbors))
            cc.extend(neighbors.tolist())
    adj = sp.coo_matrix(
        (np.ones(len(rr), dtype=np.float32), (rr, cc)),
        shape=(n, n)
    ).tocsr()
    adj = adj.maximum(adj.T)
    adj.setdiag(0)
    adj.eliminate_zeros()
    adj.data[:] = 1.0
    return adj

def normalize_graph(adj):
    adj = adj.astype(np.float32).tocsr()
    adj = adj + sp.eye(adj.shape[0], dtype=np.float32, format="csr")
    degree = np.asarray(adj.sum(axis=1)).ravel()
    d = sp.diags(np.power(np.maximum(degree, 1.0), -0.5))
    return (d @ adj @ d).tocsr()

n = len(reviews)
adj_user = normalize_graph(make_relation(reviews["user_id"].to_numpy(), n))
adj_product = normalize_graph(make_relation(reviews["prod_id"].to_numpy(), n))

def to_torch_sparse(adj):
    coo = adj.tocoo().astype(np.float32)
    indices = torch.from_numpy(
        np.vstack([coo.row, coo.col]).astype(np.int64)
    )
    values = torch.from_numpy(coo.data)
    return torch.sparse_coo_tensor(
        indices, values, coo.shape
    ).coalesce().to(device)

relations = {
    "user": to_torch_sparse(adj_user),
    "product": to_torch_sparse(adj_product)
}

semantic = torch.from_numpy(semantic_np).to(device)
labels = torch.from_numpy(reviews["target"].to_numpy(np.int64)).to(device)

class AdaptiveHybridGNN(nn.Module):
    def __init__(self, semantic_dim=384, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.user_layer1 = nn.Linear(semantic_dim, hidden_dim)
        self.product_layer1 = nn.Linear(semantic_dim, hidden_dim)
        self.graph_fusion1 = nn.Linear(hidden_dim * 2, hidden_dim)

        self.user_layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.product_layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.graph_fusion2 = nn.Linear(hidden_dim * 2, hidden_dim)

        self.semantic_projection = nn.Linear(semantic_dim, hidden_dim)
        self.gate = nn.Linear(hidden_dim * 2, 1)
        self.classifier = nn.Linear(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, semantic, relations):
        u = F.relu(self.user_layer1(torch.sparse.mm(relations["user"], semantic)))
        p = F.relu(self.product_layer1(torch.sparse.mm(relations["product"], semantic)))
        h = F.relu(self.graph_fusion1(torch.cat([u, p], dim=1)))
        h = F.dropout(h, p=self.dropout, training=self.training)

        u2 = F.relu(self.user_layer2(torch.sparse.mm(relations["user"], h)))
        p2 = F.relu(self.product_layer2(torch.sparse.mm(relations["product"], h)))
        h = self.graph_fusion2(torch.cat([u2, p2], dim=1))

        s = F.relu(self.semantic_projection(semantic))
        alpha = torch.sigmoid(self.gate(torch.cat([h, s], dim=1)))
        z = alpha * h + (1.0 - alpha) * s
        logits = self.classifier(F.dropout(z, p=self.dropout, training=self.training))
        return logits, h, s, alpha

model = AdaptiveHybridGNN().to(device)

model.eval()
with torch.no_grad():
    logits, structural, projected_semantic, alpha = model(semantic, relations)

print("Device:", device)
print("Logits:", tuple(logits.shape))
print("Structural representations:", tuple(structural.shape))
print("Projected semantic representations:", tuple(projected_semantic.shape))
print("Fusion gate:", tuple(alpha.shape))
print("Fusion gate range:", float(alpha.min()), float(alpha.max()))
print("All logits finite:", bool(torch.isfinite(logits).all()))

assert logits.shape == (10000, 2)
assert structural.shape == (10000, 128)
assert projected_semantic.shape == (10000, 128)
assert alpha.shape == (10000, 1)
assert torch.isfinite(logits).all()
print("\nHybrid model validation: PASS")


Device: cpu
Logits: (10000, 2)
Structural representations: (10000, 128)
Projected semantic representations: (10000, 128)
Fusion gate: (10000, 1)
Fusion gate range: 0.4968614876270294 0.5091789364814758
All logits finite: True

Hybrid model validation: PASS


In [50]:

import os
import copy
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

split_path = os.path.join(
    processed_dir, "yelpzip_dev_10k_splits_seed42.npz"
)
split = np.load(split_path)

train_idx = torch.as_tensor(split["train_idx"], dtype=torch.long, device=device)
val_idx = torch.as_tensor(split["validation_idx"], dtype=torch.long, device=device)
test_idx = torch.as_tensor(split["test_idx"], dtype=torch.long, device=device)

train_counts = torch.bincount(labels[train_idx], minlength=2).float()
class_weights = train_counts.sum() / (2.0 * train_counts)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.001, weight_decay=5e-4
)

def metrics_for_indices(indices):
    model.eval()
    with torch.no_grad():
        logits, _, _, _ = model(semantic, relations)
        probs = torch.softmax(logits[indices], dim=1)[:, 1].cpu().numpy()
        true = labels[indices].cpu().numpy()
        pred = (probs >= 0.5).astype(np.int64)

    return {
        "accuracy": float(accuracy_score(true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(true, pred)),
        "macro_precision": float(precision_score(
            true, pred, average="macro", zero_division=0
        )),
        "macro_recall": float(recall_score(
            true, pred, average="macro", zero_division=0
        )),
        "macro_f1": float(f1_score(
            true, pred, average="macro", zero_division=0
        )),
        "fake_f1": float(f1_score(true, pred, pos_label=1, zero_division=0)),
        "roc_auc": float(roc_auc_score(true, probs)),
        "average_precision": float(average_precision_score(true, probs)),
        "confusion_matrix": confusion_matrix(true, pred).tolist()
    }

best_val_f1 = -1.0
best_epoch = 0
best_state = None
patience = 8
stale_epochs = 0
max_epochs = 50

for epoch in range(1, max_epochs + 1):
    model.train()
    optimizer.zero_grad()

    logits, _, _, _ = model(semantic, relations)
    loss = criterion(logits[train_idx], labels[train_idx])
    loss.backward()
    optimizer.step()

    val_metrics = metrics_for_indices(val_idx)
    val_f1 = val_metrics["macro_f1"]

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        stale_epochs = 0
    else:
        stale_epochs += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d} | loss={loss.item():.4f} "
            f"| val_macro_F1={val_f1:.4f} "
            f"| best={best_val_f1:.4f} at epoch {best_epoch}"
        )

    if stale_epochs >= patience:
        print(f"Early stopping at epoch {epoch}.")
        break

model.load_state_dict(best_state)

# Evaluate the selected model on validation and test sets.
validation_results = metrics_for_indices(val_idx)
test_results = metrics_for_indices(test_idx)

# Inspect the trained fusion gate.
model.eval()
with torch.no_grad():
    _, _, _, trained_alpha = model(semantic, relations)

print("\nBest epoch:", best_epoch)
print("Validation metrics:", validation_results)
print("Test metrics:", test_results)
print(
    "Trained fusion gate — mean:",
    float(trained_alpha.mean()),
    "| std:",
    float(trained_alpha.std())
)

# Save model checkpoint and evaluation results.
results_dir = os.path.join(project_dir, "results")
os.makedirs(results_dir, exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "best_epoch": best_epoch,
        "seed": SEED,
        "embedding_model": "sentence-transformers/all-MiniLM-L6-v2"
    },
    os.path.join(results_dir, "yelpzip_hybrid_10k.pt")
)

import json
with open(os.path.join(results_dir, "yelpzip_hybrid_10k_metrics.json"), "w") as f:
    json.dump({
        "best_epoch": best_epoch,
        "validation": validation_results,
        "test": test_results
    }, f, indent=2)

print("\nSaved model and metrics.")


Epoch 01 | loss=0.6956 | val_macro_F1=0.4645 | best=0.4645 at epoch 1
Epoch 05 | loss=0.6908 | val_macro_F1=0.1877 | best=0.5807 at epoch 3
Epoch 10 | loss=0.6867 | val_macro_F1=0.5338 | best=0.5807 at epoch 3
Early stopping at epoch 11.

Best epoch: 3
Validation metrics: {'accuracy': 0.778, 'balanced_accuracy': 0.5953499241016768, 'macro_precision': 0.5739066392029124, 'macro_recall': 0.5953499241016768, 'macro_f1': 0.5806605163542399, 'fake_f1': 0.2929936305732484, 'roc_auc': 0.6677198444180936, 'average_precision': 0.23651347758529712, 'confusion_matrix': [[1098, 203], [130, 69]]}
Test metrics: {'accuracy': 0.768, 'balanced_accuracy': 0.5730112181725084, 'macro_precision': 0.5563505939835217, 'macro_recall': 0.5730112181725084, 'macro_f1': 0.5610125304852409, 'fake_f1': 0.25957446808510637, 'roc_auc': 0.6326358826358826, 'average_precision': 0.19675422678031862, 'confusion_matrix': [[1091, 211], [137, 61]]}
Trained fusion gate — mean: 0.5055102705955505 | std: 0.0013635187642648816


In [51]:

import copy
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, roc_auc_score,
    average_precision_score, confusion_matrix
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

split = np.load(
    "/content/hybrid_gnn_llm/data/processed/yelpzip_dev_10k_splits_seed42.npz"
)
train_idx = torch.as_tensor(split["train_idx"], dtype=torch.long, device=device)
val_idx = torch.as_tensor(split["validation_idx"], dtype=torch.long, device=device)
test_idx = torch.as_tensor(split["test_idx"], dtype=torch.long, device=device)

class SemanticOnly(nn.Module):
    def __init__(self, input_dim=384, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.projection = nn.Linear(input_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2)
        self.dropout = dropout

    def forward(self, semantic):
        s = F.relu(self.projection(semantic))
        s = F.dropout(s, p=self.dropout, training=self.training)
        return self.classifier(s)

semantic_model = SemanticOnly().to(device)

train_counts = torch.bincount(labels[train_idx], minlength=2).float()
weights = train_counts.sum() / (2.0 * train_counts)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(
    semantic_model.parameters(), lr=0.001, weight_decay=5e-4
)

def semantic_metrics(indices):
    semantic_model.eval()
    with torch.no_grad():
        logits = semantic_model(semantic)[indices]
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        true = labels[indices].cpu().numpy()
        pred = (probs >= 0.5).astype(np.int64)

    return {
        "accuracy": float(accuracy_score(true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(true, pred)),
        "macro_f1": float(f1_score(true, pred, average="macro", zero_division=0)),
        "fake_f1": float(f1_score(true, pred, pos_label=1, zero_division=0)),
        "roc_auc": float(roc_auc_score(true, probs)),
        "average_precision": float(average_precision_score(true, probs)),
        "confusion_matrix": confusion_matrix(true, pred).tolist()
    }

best_f1, best_state, best_epoch, stale = -1.0, None, 0, 0

for epoch in range(1, 51):
    semantic_model.train()
    optimizer.zero_grad()
    logits = semantic_model(semantic)
    loss = criterion(logits[train_idx], labels[train_idx])
    loss.backward()
    optimizer.step()

    val_f1 = semantic_metrics(val_idx)["macro_f1"]
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        best_state = copy.deepcopy(semantic_model.state_dict())
        stale = 0
    else:
        stale += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d} | loss={loss.item():.4f} "
            f"| val_macro_F1={val_f1:.4f} | best={best_f1:.4f}"
        )
    if stale >= 8:
        print("Early stopping at epoch", epoch)
        break

semantic_model.load_state_dict(best_state)
semantic_validation = semantic_metrics(val_idx)
semantic_test = semantic_metrics(test_idx)

print("\nBest epoch:", best_epoch)
print("Semantic-only validation:", semantic_validation)
print("Semantic-only test:", semantic_test)

results_dir = "/content/hybrid_gnn_llm/results"
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, "yelpzip_semantic_only_10k_metrics.json"), "w") as f:
    json.dump({
        "best_epoch": best_epoch,
        "validation": semantic_validation,
        "test": semantic_test
    }, f, indent=2)

torch.save(
    semantic_model.state_dict(),
    os.path.join(results_dir, "yelpzip_semantic_only_10k.pt")
)
print("Saved semantic-only baseline.")


Epoch 01 | loss=0.6951 | val_macro_F1=0.1180 | best=0.1180
Epoch 05 | loss=0.6889 | val_macro_F1=0.5548 | best=0.5548
Epoch 10 | loss=0.6815 | val_macro_F1=0.5756 | best=0.5756
Epoch 15 | loss=0.6726 | val_macro_F1=0.5402 | best=0.5756
Early stopping at epoch 18

Best epoch: 10
Semantic-only validation: {'accuracy': 0.756, 'balanced_accuracy': 0.6018215597588249, 'macro_f1': 0.5755754084216972, 'fake_f1': 0.2988505747126437, 'roc_auc': 0.668583115423389, 'average_precision': 0.250095087756309, 'confusion_matrix': [[1056, 245], [121, 78]]}
Semantic-only test: {'accuracy': 0.7646666666666667, 'balanced_accuracy': 0.6032095144998371, 'macro_f1': 0.5797535665390187, 'fake_f1': 0.300990099009901, 'roc_auc': 0.6784317832704929, 'average_precision': 0.23646272846786318, 'confusion_matrix': [[1071, 231], [122, 76]]}
Saved semantic-only baseline.


In [52]:

import os
import json
import pandas as pd

results_dir = "/content/hybrid_gnn_llm/results"

with open(os.path.join(results_dir, "yelpzip_hybrid_10k_metrics.json")) as f:
    hybrid = json.load(f)

with open(os.path.join(results_dir, "yelpzip_semantic_only_10k_metrics.json")) as f:
    semantic = json.load(f)

rows = []
for model_name, result in [
    ("Semantic-only MiniLM", semantic["test"]),
    ("Adaptive-fusion GNN + MiniLM", hybrid["test"])
]:
    rows.append({
        "model": model_name,
        "accuracy": result["accuracy"],
        "balanced_accuracy": result["balanced_accuracy"],
        "macro_f1": result["macro_f1"],
        "fake_f1": result["fake_f1"],
        "roc_auc": result["roc_auc"],
        "average_precision": result["average_precision"]
    })

comparison = pd.DataFrame(rows)
out_path = os.path.join(results_dir, "yelpzip_10k_model_comparison.csv")
comparison.to_csv(out_path, index=False)

print(comparison.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nSaved:", out_path)


                       model  accuracy  balanced_accuracy  macro_f1  fake_f1  roc_auc  average_precision
        Semantic-only MiniLM    0.7647             0.6032    0.5798   0.3010   0.6784             0.2365
Adaptive-fusion GNN + MiniLM    0.7680             0.5730    0.5610   0.2596   0.6326             0.1968

Saved: /content/hybrid_gnn_llm/results/yelpzip_10k_model_comparison.csv


In [53]:

import os
import json
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

project_dir = "/content/hybrid_gnn_llm"
processed_dir = os.path.join(project_dir, "data", "processed")
cache_dir = os.path.join(project_dir, "cache")
results_dir = os.path.join(project_dir, "results")

# Reload aligned reviews, labels, and semantic embeddings.
reviews = pd.read_pickle(
    os.path.join(processed_dir, "yelpzip_dev_10k.pkl")
).reset_index(drop=True)

semantic_np = np.load(
    os.path.join(cache_dir, "yelpzip_dev_10k_embeddings.npy")
).astype(np.float32)

semantic_tensor = torch.from_numpy(semantic_np).to(device)
labels = torch.from_numpy(
    reviews["target"].to_numpy(dtype=np.int64)
).to(device)

split = np.load(
    os.path.join(processed_dir, "yelpzip_dev_10k_splits_seed42.npz")
)
val_idx = split["validation_idx"]
test_idx = split["test_idx"]

assert semantic_tensor.shape == (len(reviews), 384)
assert len(labels) == len(reviews)

# Restore the saved models.
semantic_model.load_state_dict(torch.load(
    os.path.join(results_dir, "yelpzip_semantic_only_10k.pt"),
    map_location=device,
    weights_only=True
))

hybrid_checkpoint = torch.load(
    os.path.join(results_dir, "yelpzip_hybrid_10k.pt"),
    map_location=device,
    weights_only=True
)
model.load_state_dict(hybrid_checkpoint["model_state_dict"])

semantic_model.eval()
model.eval()

with torch.no_grad():
    semantic_probs = torch.softmax(
        semantic_model(semantic_tensor), dim=1
    )[:, 1].cpu().numpy()

    hybrid_logits, _, _, _ = model(semantic_tensor, relations)
    hybrid_probs = torch.softmax(
        hybrid_logits, dim=1
    )[:, 1].cpu().numpy()

y_true = labels.cpu().numpy()

def choose_threshold(probs, idx):
    thresholds = np.arange(0.10, 0.901, 0.01)
    scores = [
        f1_score(
            y_true[idx], (probs[idx] >= t).astype(int),
            average="macro", zero_division=0
        )
        for t in thresholds
    ]
    best_i = int(np.argmax(scores))
    return float(thresholds[best_i]), float(scores[best_i])

def report_at_threshold(probs, idx, threshold):
    true = y_true[idx]
    pred = (probs[idx] >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(true, pred)),
        "macro_f1": float(f1_score(
            true, pred, average="macro", zero_division=0
        )),
        "fake_f1": float(f1_score(
            true, pred, pos_label=1, zero_division=0
        )),
        "roc_auc": float(roc_auc_score(true, probs[idx])),
        "average_precision": float(average_precision_score(true, probs[idx])),
        "confusion_matrix": confusion_matrix(true, pred).tolist()
    }

rows = []
for name, probs in [
    ("Semantic-only MiniLM", semantic_probs),
    ("Adaptive-fusion GNN + MiniLM", hybrid_probs)
]:
    threshold, val_f1 = choose_threshold(probs, val_idx)
    test_metrics = report_at_threshold(probs, test_idx, threshold)

    rows.append({
        "model": name,
        "validation_selected_threshold": threshold,
        "validation_macro_f1": val_f1,
        **{f"test_{k}": v for k, v in test_metrics.items()}
    })

threshold_df = pd.DataFrame(rows)
out_path = os.path.join(
    results_dir, "yelpzip_10k_threshold_evaluation.csv"
)
threshold_df.to_csv(out_path, index=False)

print(threshold_df.to_string(index=False))
print("\nSaved:", out_path)
print("Threshold evaluation: PASS")


                       model  validation_selected_threshold  validation_macro_f1  test_threshold  test_accuracy  test_balanced_accuracy  test_macro_f1  test_fake_f1  test_roc_auc  test_average_precision    test_confusion_matrix
        Semantic-only MiniLM                            0.5             0.575575             0.5       0.764667                0.603210       0.579754      0.300990      0.678432                0.236463 [[1071, 231], [122, 76]]
Adaptive-fusion GNN + MiniLM                            0.5             0.580661             0.5       0.768000                0.573011       0.561013      0.259574      0.632636                0.196754 [[1091, 211], [137, 61]]

Saved: /content/hybrid_gnn_llm/results/yelpzip_10k_threshold_evaluation.csv
Threshold evaluation: PASS
